# Week 4 — Haalbaarheidscheck

**Data III · Leerdoelen 6 en 1**

Dit notebook is het **bewijs** bij je plan van aanpak. Het doel is niet om data te
verzamelen, maar om per bron aan te tonen dat het kán — of om nu al te ontdekken dat
het niet kan, terwijl wisselen nog goedkoop is.

> **Tien records per bron. Niet meer.**
> Dit is een *spike*: een tijdgebonden proef. Je krijgt hier maximaal 30 minuten per
> bron voor. Lukt het niet binnen die tijd, dan is dát je uitkomst — en die is
> waardevoller dan een half werkende scraper.

## Wat je per bron beantwoordt

| Vraag | Waarom |
|---|---|
| Lukt het technisch? | Sleutel, toegang, formaat |
| Zit de informatie erin die je dacht? | Een API met de juiste naam heeft niet altijd de juiste velden |
| Hoeveel records zijn er in totaal? | Bepaalt of je vraag beantwoordbaar is |
| Waar loop je op vast als het niet lukt? | Een concrete blokkade kan de docent helpen oplossen |

## Werkwijze

1. Vul eerst de **bronnenlijst** hieronder in (deel 0).
2. Doe per bron de proef in het bijbehorende deel.
3. Vul de **haalbaarheidstabel** aan het eind in.
4. Neem dit notebook mee naar de les: je laat het live zien in het go/no-go-gesprek.

## Zo werk je met dit notebook

Dit notebook gaat over **haalbaarheid van de gekozen bronnen**. Werk niet alleen naar uitvoer toe, maar volg bij ieder deel
dezelfde onderzoekscyclus:

1. **Voorspel** wat je verwacht en benoem de eenheid van de invoer en uitvoer.
2. **Voer uit** in kleine stappen; verander niet tegelijk code én testdata.
3. **Controleer** aantallen, sleutels, typen of verdelingen met een expliciete check.
4. **Verklaar** in een Markdown-cel waarom de uitkomst de verwachting wel of niet ondersteunt.
5. **Bewaar bewijs**: bewijs per bron voor toegang, dekking, koppelbaarheid, licentie en risico.

Als je vastloopt, noteer dan cel, invoer, volledige foutmelding en wat je al hebt uitgesloten.
Een screenshot zonder reproduceerbare invoer is geen diagnose.

## Opgavewijzer en hints

**Wanneer is een deel af?** Je hebt niet alleen uitvoer, maar ook (1) een voorspelling,
(2) een gerichte controle, (3) een korte conclusie en (4) bewijs van minstens één relevant
fout- of randgeval. “De cel draait” is geen inhoudelijk acceptatiecriterium.

**Als je vastloopt, gebruik deze ladder:**

1. Lees foutmelding en opgave opnieuw; noteer verwachte input, output en eenheid.
2. Verklein naar één record/bestand/pagina en print type, vorm en sleutels.
3. Vergelijk met het laatste werkende tussenresultaat; verander maar één aanname tegelijk.
4. Gebruik de hint hieronder. Vraag daarna hulp met code, invoer, verwachte uitkomst en volledige foutmelding.

> **Hint voor deze module:** Een mislukte bronproef is geldig bewijs. Noteer exacte URL/bron, tijd, response of fout en welk criterium daardoor wel of niet haalbaar is.

De hint geeft een onderzoeksrichting, niet de code. Kun je de gekozen controle niet uitleggen,
dan is het deel nog niet afgerond.

In [3]:
print("test want pushen gaat niet.")

test want pushen gaat niet.


## Setup

In [1]:
import json
import os
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

# Contactgegevens in je User-Agent zijn niet verplicht maar wel netjes: de beheerder
# van de bron kan je bereiken in plaats van je meteen te blokkeren.
USER_AGENT = "DataIII-student HHS (jouw.naam@student.hhs.nl)"
HEADERS = {"User-Agent": USER_AGENT}
TIMEOUT = 15

PROEF = Path("data/proef")
PROEF.mkdir(parents=True, exist_ok=True)

# Sleutels staan in .env (die in .gitignore staat) en komen nooit in de uitvoer.
load_dotenv(".env")
TMDB_KEY = os.environ["TMDB_API"]
OMDB_KEY = os.environ["OMDb_API"]

print("klaar — sleutels geladen:", bool(TMDB_KEY), bool(OMDB_KEY))

klaar — sleutels geladen: True True


### Hulpfuncties

Deze twee gebruik je bij elke bron.

In [2]:
def bewaar_proef(naam: str, data) -> Path:
    """Sla het resultaat van een proef op, zodat je het in de les kunt laten zien.

    Args:
        naam: korte naam van de bron, bijvoorbeeld "openmeteo".
        data: wat je hebt opgehaald (list, dict of DataFrame).

    Returns:
        Het pad waar het bestand is opgeslagen.
    """
    pad = PROEF / f"{naam}.json"
    if isinstance(data, pd.DataFrame):
        data.head(10).to_json(pad, orient="records", force_ascii=False, indent=2)
    else:
        pad.write_text(json.dumps(data[:10] if isinstance(data, list) else data,
                                  ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"proef bewaard: {pad}")
    return pad


def vat_samen(df: pd.DataFrame, naam: str) -> None:
    """Print een compacte samenvatting van een proefresultaat."""
    print(f"--- {naam} ---")
    print(f"vorm      : {df.shape[0]} rijen x {df.shape[1]} kolommen")
    print(f"kolommen  : {list(df.columns)}")
    print(f"lege cellen per kolom:\n{df.isna().sum().to_string()}")


def haal_json(url: str, params: dict | None = None,
              sleutel_param: str | None = None, sleutel: str | None = None) -> dict:
    """GET-verzoek dat JSON teruggeeft, zonder de API-sleutel in foutmeldingen te lekken.

    requests zet de volledige URL (inclusief ?api_key=...) in zijn foutmeldingen.
    Daarom vangen we die af en geven we een eigen melding zonder sleutel.
    """
    volledige_params = dict(params or {})
    if sleutel_param:
        volledige_params[sleutel_param] = sleutel
    try:
        antwoord = requests.get(url, params=volledige_params, headers=HEADERS, timeout=TIMEOUT)
    except requests.RequestException as fout:
        raise RuntimeError(f"verbinding mislukt bij {url}: {type(fout).__name__}") from None
    if antwoord.status_code != 200:
        raise RuntimeError(f"status {antwoord.status_code} bij {url} (params: {params})")
    return antwoord.json()


def tmdb(pad: str, **params) -> dict:
    """Roep een TMDb v3-endpoint aan, bijvoorbeeld tmdb("/movie/550")."""
    return haal_json(f"https://api.themoviedb.org/3{pad}", params, "api_key", TMDB_KEY)


def haal_json(url: str, params: dict | None = None,
              sleutel_param: str | None = None, sleutel: str | None = None) -> dict:
    """GET-verzoek dat JSON teruggeeft, zonder de API-sleutel in foutmeldingen te lekken.

    requests zet de volledige URL (inclusief ?api_key=...) in zijn foutmeldingen.
    Daarom vangen we die af en geven we een eigen melding zonder sleutel.
    """
    volledige_params = dict(params or {})
    if sleutel_param:
        volledige_params[sleutel_param] = sleutel
    try:
        antwoord = requests.get(url, params=volledige_params, headers=HEADERS, timeout=TIMEOUT)
    except requests.RequestException as fout:
        raise RuntimeError(f"verbinding mislukt bij {url}: {type(fout).__name__}") from None
    if antwoord.status_code != 200:
        raise RuntimeError(f"status {antwoord.status_code} bij {url} (params: {params})")
    return antwoord.json()


def tmdb(pad: str, **params) -> dict:
    """Roep een TMDb v3-endpoint aan, bijvoorbeeld tmdb("/movie/550")."""
    return haal_json(f"https://api.themoviedb.org/3{pad}", params, "api_key", TMDB_KEY)

---

# Deel 0 — Je bronnenlijst

Vul deze tabel in vóórdat je begint. Alles wat je hier niet concreet kunt maken, is een
signaal: "een API met filmdata" is geen bron, "The Movie Database, endpoint
`/discover/movie`" wel.

| # | Bron (naam + URL/endpoint) | Deelvraag | Techniek | Modaliteit | Sleutel nodig? |
|---|---|---|---|---|---|
| 1 | **TMDb API v3** — `GET /3/discover/movie` + `GET /3/movie/{id}?append_to_response=reviews`; aanvulling **OMDb** `GET https://www.omdbapi.com/?i={imdb_id}` | DV2 (genre, budget, speelduur, releasejaar) en DV3 (reviews → sentiment) | API | tabel + tekst | ja — TMDb- en OMDb-sleutel in bezit (staan in `.env`) |
| 2 | **Wikipedia (en)** filmpagina, infobox — `https://en.wikipedia.org/wiki/<Titel>_(<jaar>_film)` | succesmaat: budget en box office als onafhankelijke aanvulling op TMDb | scraping | tabel (semi-gestructureerde HTML) | nee |
| 3 | **Wikidata Query Service** — SPARQL op `https://query.wikidata.org/sparql` | DV1: koppel-id's (P4947 TMDb, P345 IMDb) + budget (P2130) en box office (P2142) | database | tabel | nee |
| 4 | **TMDb-posters** — `https://image.tmdb.org/t/p/w342{poster_path}` | DV4: visuele kenmerken van de poster | API (bestanden) | beeld | nee (pad komt uit bron 1) |
| 5 | **Deezer API** — `GET /search/album`, `/album/{id}`, `/album/{id}/tracks`, 30-s `preview` | DV5: audio-eigenschappen van de soundtrack | API | audio | nee |

**Eenheid van analyse:** één rij in mijn eindtabel is één **film** (uniek via `tmdb_id`),
uitgebracht 2010–2019 met ≥ 50 stemmen op TMDb.

**Verwacht aantal rijen in de eindtabel:** vooraf geschat op enkele duizenden. De proef laat
zien dat dat met deze afbakening **niet** klopt (zie deel 6 en de uitdaging): bij een inner
join over alle modaliteiten blijft er in de steekproef 0 van 10 over. Met de aangepaste
afbakening (≥ 1000 stemmen, 1.843 films) verwachten we ≈ 900 complete rijen.

---

# Deel 1 — Je API-bron

### Bron 1 — TMDb API (met OMDb als aanvulling)

| | |
|---|---|
| **Bron** | TMDb API v3: `/3/discover/movie` (selectie + telling), `/3/movie/{id}?append_to_response=reviews` (details + reviews). Aanvulling: OMDb `/?i={imdb_id}` |
| **Deelvraag** | DV2 (genre, budget, speelduur, releasejaar) en DV3 (reviewtekst → sentiment); levert ook de joinsleutels `tmdb_id` en `imdb_id` |
| **Techniek** | API |
| **Modaliteit** | tabel + tekst (reviews) |
| **Sleutel nodig?** | ja — TMDb- en OMDb-sleutel zijn al in bezit en staan in `.env` (in `.gitignore`) |

**Wat je hier moet aantonen:** tien echte records, en dat de velden erin zitten die je
voor je deelvraag nodig hebt.

**Voorspelling (vóór het draaien).** Binnen 2010–2019 met ≥ 50 stemmen verwachten we
ruim 10.000 films. Genre, speelduur en jaar zitten bij alle tien de films. Budget en
opbrengst verwachten we bij hooguit de helft: TMDb vult `0` in als het bedrag onbekend is,
en voor kleine films weet niemand het. Reviews zijn schaars bij minder bekende films.
Eenheden: 1 record = 1 film; bedragen in nominale USD; speelduur in minuten.

**Werkwijze.** We nemen bewust níet de eerste tien (dat zijn de populairste of oudste
films, afhankelijk van de sortering), maar tien willekeurige pagina's met uit elke pagina
één willekeurige film. Zo is de dekking die we meten een eerlijke schatting.

In [3]:
# Stap 1 — hoeveel films vallen er binnen onze afbakening, en welke tien nemen we?
FILTER = {
    "primary_release_date.gte": "2010-01-01",
    "primary_release_date.lte": "2019-12-31",
    "vote_count.gte": 50,          # alleen films waar genoeg over bekend is
    "include_adult": "false",
    "language": "en-US",
}
SORTERING = "primary_release_date.asc"   # stabiel; popularity verschuift per dag

eerste = tmdb("/discover/movie", **FILTER, sort_by=SORTERING, page=1)
TOTAAL_TMDB = eerste["total_results"]
print(f"films binnen de afbakening: {TOTAAL_TMDB} ({eerste['total_pages']} pagina's)")
print("velden in een zoekresultaat:", list(eerste["results"][0].keys()))

# Geen 'eerste tien': 10 willekeurige pagina's, uit elke pagina 1 willekeurige film.
# De gekozen id's bewaren we, zodat een herhaalde run dezelfde steekproef geeft.
steekproef_pad = PROEF / "steekproef_tmdb_ids.json"
if steekproef_pad.exists():
    steekproef = json.loads(steekproef_pad.read_text())
else:
    rng = random.Random(42)
    paginas = rng.sample(range(1, min(eerste["total_pages"], 500) + 1), 10)
    steekproef = []
    for pagina in paginas:
        resultaten = tmdb("/discover/movie", **FILTER, sort_by=SORTERING, page=pagina)["results"]
        steekproef.append(rng.choice(resultaten)["id"])
        time.sleep(0.25)
    steekproef_pad.write_text(json.dumps(steekproef))
print("steekproef (tmdb_id):", steekproef)

films binnen de afbakening: 12084 (605 pagina's)
velden in een zoekresultaat: ['adult', 'backdrop_path', 'genre_ids', 'id', 'title', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'release_date', 'softcore', 'video', 'vote_average', 'vote_count']
steekproef (tmdb_id): [348320, 56780, 11761, 345925, 149786, 127372, 101267, 84111, 296867, 51209]


In [4]:
# Stap 2 — per film de details + reviews (tekstmodaliteit) in één aanroep.
rijen = []
for tmdb_id in steekproef:
    f = tmdb(f"/movie/{tmdb_id}", append_to_response="reviews")
    reviews = f["reviews"]["results"]
    rijen.append({
        "tmdb_id": f["id"],
        "imdb_id": f.get("imdb_id") or None,
        "titel": f["title"],
        "jaar": int(f["release_date"][:4]),
        "genres": "|".join(g["name"] for g in f["genres"]),
        "runtime_min": f["runtime"] or None,
        "budget_ruw": f["budget"],               # TMDb gebruikt 0 voor 'onbekend'
        "budget_usd": f["budget"] or None,
        "revenue_usd": f["revenue"] or None,
        "vote_average": f["vote_average"],
        "vote_count": f["vote_count"],
        "poster_path": f["poster_path"],
        "n_reviews": f["reviews"]["total_results"],
        # alleen de tekst, niet de auteursnaam (persoonsgegeven, niet nodig)
        "review_tekst": reviews[0]["content"][:400] if reviews else None,
    })
    time.sleep(0.25)

df_api = pd.DataFrame(rijen)
vat_samen(df_api, "TMDb")
df_api.drop(columns=["review_tekst"])

--- TMDb ---
vorm      : 10 rijen x 14 kolommen
kolommen  : ['tmdb_id', 'imdb_id', 'titel', 'jaar', 'genres', 'runtime_min', 'budget_ruw', 'budget_usd', 'revenue_usd', 'vote_average', 'vote_count', 'poster_path', 'n_reviews', 'review_tekst']
lege cellen per kolom:
tmdb_id         0
imdb_id         0
titel           0
jaar            0
genres          0
runtime_min     0
budget_ruw      0
budget_usd      8
revenue_usd     6
vote_average    0
vote_count      0
poster_path     0
n_reviews       0
review_tekst    7


,tmdb_id,imdb_id,titel,jaar,genres,runtime_min,budget_ruw,budget_usd,revenue_usd,vote_average,vote_count,poster_path,n_reviews
0,348320,tt4849176,Scream Week,2016,Comedy|Thriller|Horror,113,0,NaN,NaN,4.900,71,/biVZIlu3vyo2ToZjVc6daBveMMG.jpg,0
1,56780,tt0480239,Atlas Shrugged: Part I,2011,Drama|Thriller|Science Fiction|Mystery,97,20000000,20000000.0,4627375.0,5.473,222,/tYfZIQb6ZabSUy1s1TRqSrYaAoM.jpg,0
2,11761,tt1188113,Peacock,2010,Mystery|Thriller|Drama,90,0,NaN,NaN,6.122,263,/iEtrsxfNw9qmAeML3QSzOe0hrS0.jpg,0
3,345925,tt4578118,Rupture,2016,Thriller|Science Fiction,102,0,NaN,NaN,4.918,297,/41MCf9bzSpqhr7EAu95su0XdjVQ.jpg,1
4,149786,tt2355762,Christmas with Holly,2012,Drama|Family|TV Movie|Romance,90,0,NaN,NaN,6.300,100,/bGbvMw4pkSHLFAYVjvOZsR0WwPZ.jpg,0
5,127372,tt2103264,Emperor,2012,Drama|History|War,105,0,NaN,14858240.0,6.155,290,/lKcRXcffQM2KxNruljrd7H3gUe5.jpg,1
6,101267,tt2215719,Katy Perry: Part of Me,2012,Documentary|Music,93,12000000,12000000.0,32726956.0,7.100,291,/yPW5Lmptt7z8rCYtVYjkpyjI3SY.jpg,0
7,84111,tt2036416,You Are the Apple of My Eye,2011,Romance|Comedy|Drama,109,0,NaN,NaN,7.400,241,/mQ07Y4KlYZlJjVSEp2AFTYZA9i.jpg,0
8,296867,tt4457020,El Jeremías,2016,Family|Comedy|Drama,98,0,NaN,NaN,6.867,60,/iLJ817Xa8TuZsGRghuPmtkVdV4k.jpg,0
9,51209,tt1416801,Kill the Irishman,2011,Action|Crime|Thriller|Drama,106,0,NaN,1188194.0,6.785,730,/yjDvqAMyFlesBtWNUUZMgglo78y.jpg,1


In [5]:
# Stap 3 — expliciete controles in plaats van "de cel draait".
assert len(df_api) == 10, "precies tien records verwacht"
assert df_api["tmdb_id"].is_unique, "tmdb_id moet uniek zijn (granulariteit: 1 rij = 1 film)"
assert df_api["jaar"].between(2010, 2019).all(), "filter op releasejaar werkt niet"

print("budget = 0 in de ruwe data (betekent 'onbekend'):", (df_api["budget_ruw"] == 0).sum(), "/ 10")
print("budget bekend      :", df_api["budget_usd"].notna().sum(), "/ 10")
print("revenue bekend     :", df_api["revenue_usd"].notna().sum(), "/ 10")
print("imdb_id bekend     :", df_api["imdb_id"].notna().sum(), "/ 10")
print("≥1 review (tekst)  :", (df_api["n_reviews"] > 0).sum(), "/ 10")
print("typen:", df_api[["jaar", "runtime_min", "budget_usd", "vote_average"]].dtypes.to_dict())

# Randgeval: een id die niet bestaat moet een nette fout geven (en geen sleutel lekken).
try:
    tmdb("/movie/0")
except RuntimeError as fout:
    print("randgeval /movie/0 ->", fout)

print()
print("voorbeeld reviewtekst:", (df_api["review_tekst"].dropna().iloc[0][:200] + " …")
      if df_api["review_tekst"].notna().any() else "geen")

budget = 0 in de ruwe data (betekent 'onbekend'): 8 / 10
budget bekend      : 2 / 10
revenue bekend     : 4 / 10
imdb_id bekend     : 10 / 10
≥1 review (tekst)  : 3 / 10
typen: {'jaar': dtype('int64'), 'runtime_min': dtype('int64'), 'budget_usd': dtype('float64'), 'vote_average': dtype('float64')}


randgeval /movie/0 -> status 404 bij https://api.themoviedb.org/3/movie/0 (params: {})

voorbeeld reviewtekst: Hammy acting, terrible dialogue, nonsensical plot that thinks it is being mysterious but is actually obtuse. I love a good ambiguous film that leaves it up to the viewer to unravel, but this film simu …


In [6]:
# Stap 4 — OMDb als tweede API op dezelfde films, gekoppeld via imdb_id.
omdb_rijen = []
for imdb_id in df_api["imdb_id"].dropna():
    o = haal_json("https://www.omdbapi.com/", {"i": imdb_id}, "apikey", OMDB_KEY)
    omdb_rijen.append({
        "imdb_id": imdb_id,
        "omdb_gevonden": o.get("Response") == "True",
        "box_office_us": o.get("BoxOffice"),
        "metascore": o.get("Metascore"),
        "imdb_rating": o.get("imdbRating"),
        "imdb_votes": o.get("imdbVotes"),
    })
df_omdb = pd.DataFrame(omdb_rijen).replace("N/A", pd.NA)
vat_samen(df_omdb, "OMDb")

# Randgeval: OMDb geeft bij een onbekend id óók status 200 — alleen 'Response' verraadt het.
onbekend = haal_json("https://www.omdbapi.com/", {"i": "tt0000000"}, "apikey", OMDB_KEY)
print("randgeval tt0000000 ->", onbekend)

bewaar_proef("api_tmdb", df_api)
bewaar_proef("api_omdb", df_omdb)
df_omdb

--- OMDb ---
vorm      : 10 rijen x 6 kolommen
kolommen  : ['imdb_id', 'omdb_gevonden', 'box_office_us', 'metascore', 'imdb_rating', 'imdb_votes']
lege cellen per kolom:
imdb_id          0
omdb_gevonden    0
box_office_us    5
metascore        5
imdb_rating      0
imdb_votes       0


randgeval tt0000000 -> {'Response': 'False', 'Error': 'Error getting data.'}
proef bewaard: data\proef\api_tmdb.json
proef bewaard: data\proef\api_omdb.json


,imdb_id,omdb_gevonden,box_office_us,metascore,imdb_rating,imdb_votes
0,tt4849176,True,NaN,NaN,4.6,"1,801"
1,tt0480239,True,"$4,627,375",28,5.6,"14,697"
2,tt1188113,True,NaN,NaN,6.2,"13,422"
3,tt4578118,True,NaN,35,4.9,"7,876"
4,tt2355762,True,NaN,NaN,7.1,"3,809"
5,tt2103264,True,"$3,346,265",48,6.5,"15,430"
6,tt2215719,True,"$25,326,071",57,6.0,"14,022"
7,tt2036416,True,NaN,NaN,7.5,"12,308"
8,tt4457020,True,"$650,465",NaN,7.2,"1,294"
9,tt1416801,True,"$1,188,194",50,7.0,"53,953"


**Vul in na de proef:**

- [x] Tien records opgehaald — 10 films, aselect uit 12.084 (TMDb) en dezelfde 10 via OMDb (10/10 gevonden)
- [x] De velden die ik nodig heb zitten erin: genres, runtime, release_date, budget, revenue,
  vote_average/vote_count, poster_path, imdb_id en reviewtekst. **Maar** ze zijn vaak leeg:
  budget 2/10 (8× `0` = onbekend), revenue 4/10, ≥ 1 review 3/10. OMDb: BoxOffice (alleen VS) 5/10, Metascore 5/10.
- [x] Totaal beschikbaar aantal records: **12.084** films (2010–2019, ≥ 50 stemmen; `total_results`
  van `/discover/movie`), en 1.843 als we ≥ 1000 stemmen eisen.
- [x] `robots.txt` / voorwaarden nagekeken: [TMDb API Terms of Use](https://www.themoviedb.org/api-terms-of-use)
  (gelezen 23-09-2026): alleen niet-commercieel gebruik, bronvermelding verplicht ("This product uses
  TMDB and the TMDB APIs but is not endorsed, certified, or otherwise approved by TMDB"), en het
  trainen of valideren van een ML-/AI-systeem met TMDb-content geldt als commercieel gebruik.
  OMDb: *"All content licensed under CC BY-NC 4.0"* ([omdbapi.com](https://www.omdbapi.com/), 23-09-2026).
- [x] Persoonsgegevens in deze bron: **ja** — reviews hebben een auteursnaam en avatar. Die hebben we
  niet nodig en slaan we niet op; we bewaren alleen de reviewtekst.

*Conclusie:* Technisch **go**: sleutel, toegang en formaat werken. We hadden verwacht dat
budget en opbrengst bij de helft zouden ontbreken, maar bij een willekeurige film ontbreekt het
budget in 8 van de 10 gevallen. Dat valt tegen. De tekstmodaliteit is ook dun (3/10 heeft een
review). Randgevallen: `/movie/0` geeft een nette 404-melding zonder de sleutel te lekken.
OMDb geeft bij een onbekend id óók status 200, met `Response: False`. Alleen op de statuscode
controleren is daar dus niet genoeg.

---

# Deel 2 — Je scraping-bron

### Bron 2 — Wikipedia (Engelstalige filmpagina's)

| | |
|---|---|
| **Bron** | `https://en.wikipedia.org/wiki/<Titel>_(<jaar>_film)` — de infobox rechtsboven |
| **Deelvraag** | succesmaat (budget, box office) als onafhankelijke aanvulling op en controle van TMDb |
| **Techniek** | scraping |
| **Modaliteit** | tabel (semi-gestructureerde HTML) |
| **Sleutel nodig?** | nee — en ook geen login (zie blokkade in deel 6) |

**Wat je hier moet aantonen:** tien echte records, en dat de velden erin zitten die je
voor je deelvraag nodig hebt.

> Letterboxd (uit het PvA) valt af: de [Terms of Use](https://letterboxd.com/legal/terms-of-use/)
> verbieden *"any robot, spider, scraper … or other automated data gathering"* (gelezen 23-09-2026).
> Dat weegt zwaarder dan `robots.txt`, dat de losse film- en reviewpagina's wél toestaat.

> ⚠️ **Kijk eerst naar `robots.txt` en de gebruiksvoorwaarden.** Week 5 gaat hier
> uitgebreid op in, maar voor deze proef geldt al: haal één pagina op, niet honderd, en
> zet er een `User-Agent` met contactgegevens op.

Voor de proef is het genoeg om aan te tonen dat:

1. je de pagina kunt ophalen (statuscode 200, en geen blokkadepagina);
2. de gegevens die je zoekt daadwerkelijk in de HTML staan — en niet pas door
   JavaScript worden ingeladen (dan heb je week 6 nodig).

**Voorspelling.** Wikipedia bouwt de pagina op de server op, dus de infobox staat in de
ruwe HTML (geen JavaScript nodig). We verwachten voor 8 à 9 van de 10 films een Engelstalige
pagina. Bedragen staan er als vrije tekst in, soms in een andere valuta. Om zeker te weten dat
we de juiste film hebben (en niet het boek of een gelijknamige film), accepteren we een pagina
alleen als die naar hetzelfde IMDb-id linkt.

In [7]:
# Stap 1 — mag het?
from urllib.robotparser import RobotFileParser

domein = "https://en.wikipedia.org"
robots = requests.get(f"{domein}/robots.txt", headers=HEADERS, timeout=TIMEOUT)
print(robots.status_code, "|", len(robots.text), "tekens")

parser = RobotFileParser()
parser.parse(robots.text.splitlines())
for pad in ["/wiki/Inception", "/wiki/Special:Search", "/w/index.php?title=Inception"]:
    print(f"mag {pad:32}:", parser.can_fetch(USER_AGENT, domein + pad))

200 | 28028 tekens
mag /wiki/Inception                 : True
mag /wiki/Special:Search            : False
mag /w/index.php?title=Inception    : False


In [8]:
# Stap 2 — komt de data mee in de HTML, of pas via JavaScript?
from bs4 import BeautifulSoup

url = f"{domein}/wiki/Inception"
pagina = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
print("status:", pagina.status_code, "| lengte:", len(pagina.text), "tekens")

# In de browser staat in de infobox 'Box office'. Staat dat ook in de ruwe HTML?
for zoekterm in ["Box office", "Budget", "tt1375666"]:
    print(f"{zoekterm!r:14} staat in de HTML:", zoekterm in pagina.text)

status: 200 | lengte: 996545 tekens
'Box office'   staat in de HTML: True
'Budget'       staat in de HTML: True
'tt1375666'    staat in de HTML: True


> **Deze test is belangrijker dan hij lijkt.** Als je zoekterm niet in `pagina.text`
> staat, is de pagina met JavaScript opgebouwd en werkt BeautifulSoup niet. Dan heb je
> Selenium of Playwright nodig (week 6) — of, beter, de verborgen JSON-API die de pagina
> zelf aanroept. Die vind je in de developer tools onder *Network → Fetch/XHR*.

In [9]:
# Stap 3 — tien records: de infobox van de Wikipedia-pagina van elke film uit bron 1.

def infobox(html: str) -> dict:
    """Lees de infobox (label -> tekst) uit een Wikipedia-pagina."""
    soep = BeautifulSoup(html, "html.parser")
    box = soep.select_one("table.infobox")
    if box is None:
        return {}
    waarden = {}
    for tr in box.select("tr"):
        th, td = tr.find("th"), tr.find("td")
        if th and td:
            for voetnoot in td.select("sup"):
                voetnoot.decompose()
            waarden[th.get_text(" ", strip=True)] = td.get_text(" ", strip=True)
    return waarden


def geld_naar_usd(tekst: str | None) -> float | None:
    """'$160 million' -> 160000000.0. Bij een bereik ('$150–185 million') de ondergrens.
    Alleen US-dollars: 'NT $50 million', 'A$3 million' of '£8 million' -> overslaan,
    maar een omrekening als '(approx. US$1.7 million)' wordt wél gebruikt."""
    if not tekst:
        return None
    patroon = r"([A-Z]{0,3})\s?\$\s*([\d.,]+)\s*(?:[–-]\s*[\d.,]+\s*)?(million|billion)?"
    for valuta, getal, schaal in re.findall(patroon, tekst):
        if valuta in ("", "US"):
            factor = {"million": 1e6, "billion": 1e9}.get(schaal, 1)
            return float(getal.replace(",", "").rstrip(".")) * factor
    return None


def zoek_wikipagina(titel: str, jaar: int, imdb_id: str | None):
    """Probeer de gebruikelijke paginatitels; accepteer alleen de pagina die naar
    hetzelfde IMDb-id linkt (zo pakken we niet het boek of de gelijknamige film)."""
    for kandidaat in [f"{titel} ({jaar} film)", f"{titel} (film)", titel]:
        u = f"{domein}/wiki/" + kandidaat.replace(" ", "_")
        p = requests.get(u, headers=HEADERS, timeout=TIMEOUT)
        time.sleep(1)   # één pagina per seconde
        if p.status_code == 200 and imdb_id and imdb_id in p.text:
            return u, p.text
    return None, None


wiki_rijen = []
for film in df_api.itertuples():
    u, html = zoek_wikipagina(film.titel, film.jaar, film.imdb_id)
    box = infobox(html) if html else {}
    wiki_rijen.append({
        "tmdb_id": film.tmdb_id,
        "wiki_url": u,
        "budget_tekst": box.get("Budget"),
        "box_office_tekst": box.get("Box office"),
        "budget_usd_wiki": geld_naar_usd(box.get("Budget")),
        "box_office_usd_wiki": geld_naar_usd(box.get("Box office")),
    })

df_wiki = pd.DataFrame(wiki_rijen)
vat_samen(df_wiki, "Wikipedia")
print("pagina gevonden:", df_wiki["wiki_url"].notna().sum(), "/ 10")

# Randgeval-controles op de parser
for voorbeeld in ["$160 million", "$150–185 million", "£8 million", "$1.2 billion",
                  "NT $50 million (approx. US$1.7 million)", "A$3 million", None]:
    print(f"{str(voorbeeld):42} -> {geld_naar_usd(voorbeeld)}")

bewaar_proef("scraping_wikipedia", df_wiki)
df_wiki.drop(columns=["wiki_url"])

--- Wikipedia ---
vorm      : 10 rijen x 6 kolommen
kolommen  : ['tmdb_id', 'wiki_url', 'budget_tekst', 'box_office_tekst', 'budget_usd_wiki', 'box_office_usd_wiki']
lege cellen per kolom:
tmdb_id                0
wiki_url               1
budget_tekst           6
box_office_tekst       4
budget_usd_wiki        6
box_office_usd_wiki    4
pagina gevonden: 9 / 10
$160 million                               -> 160000000.0
$150–185 million                           -> 150000000.0
£8 million                                 -> None
$1.2 billion                               -> 1200000000.0
NT $50 million (approx. US$1.7 million)    -> 1700000.0
A$3 million                                -> None
None                                       -> None
proef bewaard: data\proef\scraping_wikipedia.json


,tmdb_id,budget_tekst,box_office_tekst,budget_usd_wiki,box_office_usd_wiki
0,348320,NaN,NaN,NaN,NaN
1,56780,$20 million,$4.6 million,20000000.0,4600000.0
2,11761,NaN,NaN,NaN,NaN
3,345925,NaN,"$31,238",NaN,31238.0
4,149786,NaN,NaN,NaN,NaN
5,127372,NaN,$14.9 million,NaN,14900000.0
6,101267,$13 million,$32.7 million,13000000.0,32700000.0
7,84111,NT $50 million (approx. US$1.7 million),US$ 24.5 million,1700000.0,24500000.0
8,296867,NaN,NaN,NaN,NaN
9,51209,$12 million,$1.2 million,12000000.0,1200000.0


**Vul in na de proef:**

- [x] Tien records opgehaald — 10 films geprobeerd, voor **9/10** een pagina met het juiste IMDb-id gevonden
- [x] De velden die ik nodig heb zitten erin: `Budget` bij 4/10, `Box office` bij 6/10 (als tekst,
  omgezet naar USD). Randgeval gevonden: *"NT $50 million (approx. US$1.7 million)"* las de eerste
  parser als 50 miljoen USD. De parser herkent nu het valutavoorvoegsel en gebruikt de omrekening naar US$.
- [x] Totaal beschikbaar aantal records: de site heeft geen telvraag. In de proef had 9 van de 10 films
  een pagina, dus we schatten ≈ 90 % van de films. Wikidata (bron 3) kan het exact tellen via de sitelinks.
- [x] `robots.txt` / voorwaarden nagekeken: [robots.txt](https://en.wikipedia.org/robots.txt) staat `/wiki/`
  toe en verbiedt `/wiki/Special:` en `/w/index.php` (hierboven met `RobotFileParser` gecontroleerd).
  Tekst valt onder CC BY-SA 4.0; wij nemen alleen feiten (bedragen) over. Het
  [User-Agent-beleid van Wikimedia](https://meta.wikimedia.org/wiki/User-Agent_policy) vraagt contactgegevens in de User-Agent.
  We halen maximaal één pagina per seconde op.
- [x] Persoonsgegevens in deze bron: nee — we lezen alleen budget en box office (cast staat er wel, maar slaan we niet op).

*Conclusie:* **Go** voor de techniek scraping. De data staat in de HTML en BeautifulSoup is genoeg.
Als succesmaat vult Wikipedia TMDb goed aan: TMDb had bij deze tien films 2 budgetten, alle bronnen
samen 4. **Blokkade:** de eis *"webscraping met login op de bron"* wordt hiermee **niet** gedekt.

---

# Deel 3 — Je database-bron

### Bron 3 — Wikidata Query Service (SPARQL)

| | |
|---|---|
| **Bron** | `https://query.wikidata.org/sparql` — SPARQL op de RDF-graafdatabase (triplestore) achter Wikidata |
| **Deelvraag** | DV1: koppeling tussen de bronnen (P4947 TMDb-id, P345 IMDb-id, Wikipedia-artikel) + budget (P2130) en box office (P2142) met valuta en regio |
| **Techniek** | database |
| **Modaliteit** | tabel |
| **Sleutel nodig?** | nee |

**Wat je hier moet aantonen:** tien echte records, en dat de velden erin zitten die je
voor je deelvraag nodig hebt.

> **Let op:** een gedownload CSV-bestand van Kaggle is **geen** database-extractie. De
> eis is dat je een query draait op een relationele of niet-relationele database. Week 7
> (SQL) en de weken 8–9 (NoSQL) gaan hierover.

Voor de proef toon je aan dat je verbinding kunt maken en één query kunt draaien.
Mogelijke bronnen:

* een publieke read-only PostgreSQL/MySQL-database (bijvoorbeeld van een open-datainitiatief);
* een SPARQL- of REST-endpoint boven op een database;
* een publieke MongoDB-instantie;
* een gepubliceerd SQLite- of DuckDB-bestand dat je zelf bevraagt.

Weet je nog niet welke, kies er dan één en probeer het nú — dit is de bron waar de
meeste groepen op vastlopen.

**Waarom Wikidata en niet MovieLens?** MovieLens is een CSV-download. Volgens de opgave is
dat geen database-extractie. MovieLens in SQLite laden en daarop queryen is een mogelijk plan B,
maar dan vullen we de database zelf. Wikidata is een echte publieke database waarop we een
query draaien. De SPARQL-endpoint staat in de lijst hierboven als geldige optie.

**Voorspelling.** Via het TMDb-id vinden we alle tien de films. Budget en box office verwachten
we bij een minderheid. Een film kan meerdere box-office-stellingen hebben (VS, wereldwijd,
per land), en dan geeft de query **meer rijen dan films**. Daarom tellen we rijen per film en
aggregeren we terug naar één rij per film.

In [10]:
# Wikidata Query Service: SPARQL op de Wikibase-graafdatabase achter Wikidata.
SPARQL = "https://query.wikidata.org/sparql"


def sparql(query: str, timeout: int = 60) -> pd.DataFrame:
    """Voer een SPARQL-query uit en geef het resultaat als DataFrame.
    Bij 429 (te veel verzoeken) wachten we de opgegeven Retry-After en proberen één keer opnieuw."""
    for poging in range(2):
        antwoord = requests.get(SPARQL, params={"query": query, "format": "json"},
                                headers=HEADERS, timeout=timeout)
        if antwoord.status_code != 429:
            break
        wacht = int(antwoord.headers.get("Retry-After", 30))
        print(f"429 van WDQS, {wacht} s wachten …")
        time.sleep(wacht)
    if antwoord.status_code != 200:
        raise RuntimeError(f"status {antwoord.status_code}: {antwoord.text[:200]}")
    rijen = antwoord.json()["results"]["bindings"]
    return pd.DataFrame([{k: v["value"] for k, v in rij.items()} for rij in rijen])


waarden = " ".join(f'"{i}"' for i in df_api["tmdb_id"])
query = f"""
SELECT ?tmdb ?film ?imdb ?enwiki ?budget ?budgetValuta ?bo ?boValuta ?boRegio WHERE {{
  VALUES ?tmdb {{ {waarden} }}
  ?film wdt:P4947 ?tmdb .                                   # TMDb movie ID
  OPTIONAL {{ ?film wdt:P345 ?imdb }}                        # IMDb ID
  OPTIONAL {{ ?enwiki schema:about ?film ;
                     schema:isPartOf <https://en.wikipedia.org/> }}
  OPTIONAL {{ ?film p:P2130/psv:P2130 [ wikibase:quantityAmount ?budget ;
                                       wikibase:quantityUnit   ?budgetValuta ] }}
  OPTIONAL {{ ?film p:P2142 ?boStelling .                    # box office
             ?boStelling psv:P2142 [ wikibase:quantityAmount ?bo ;
                                     wikibase:quantityUnit   ?boValuta ] .
             OPTIONAL {{ ?boStelling pq:P3005 ?boRegio }} }}  # 'geldig in' (bv. wereldwijd)
}}"""
df_db_ruw = sparql(query)
vat_samen(df_db_ruw, "Wikidata (ruw)")
print("\nrijen per film (let op: >1 = meerdere budget/box-office-stellingen):")
print(df_db_ruw.groupby("tmdb")["film"].size().to_string())

--- Wikidata (ruw) ---
vorm      : 10 rijen x 8 kolommen
kolommen  : ['film', 'bo', 'boValuta', 'budget', 'budgetValuta', 'enwiki', 'tmdb', 'imdb']
lege cellen per kolom:
film            0
bo              7
boValuta        7
budget          8
budgetValuta    8
enwiki          1
tmdb            0
imdb            0

rijen per film (let op: >1 = meerdere budget/box-office-stellingen):
tmdb
101267    1
11761     1
127372    1
149786    1
296867    1
345925    1
348320    1
51209     1
56780     1
84111     1


In [11]:
# Terug naar granulariteit 'één rij = één film': alleen USD-bedragen, hoogste waarde.
USD = "http://www.wikidata.org/entity/Q4917"
db = df_db_ruw.copy()
for kol in ["budget", "bo"]:
    if kol not in db:
        db[kol] = pd.NA
    db[kol] = pd.to_numeric(db[kol], errors="coerce")
db["budget_usd_wd"] = db["budget"].where(db.get("budgetValuta") == USD)
db["box_office_usd_wd"] = db["bo"].where(db.get("boValuta") == USD)

df_db = (db.groupby("tmdb", as_index=False)
           .agg(wikidata_id=("film", "first"), imdb_id_wd=("imdb", "first"),
                budget_usd_wd=("budget_usd_wd", "max"),
                box_office_usd_wd=("box_office_usd_wd", "max"),
                n_stellingen=("film", "size"))
           .rename(columns={"tmdb": "tmdb_id"}))
df_db["tmdb_id"] = df_db["tmdb_id"].astype(int)
df_db["wikidata_id"] = df_db["wikidata_id"].str.rsplit("/", n=1).str[-1]

assert df_db["tmdb_id"].is_unique
ontbreekt = set(df_api["tmdb_id"]) - set(df_db["tmdb_id"])
print("films uit de steekproef gevonden in Wikidata:", len(df_db), "/ 10; ontbrekend:", ontbreekt or "geen")
print("budget (USD) bekend    :", df_db["budget_usd_wd"].notna().sum())
print("box office (USD) bekend:", df_db["box_office_usd_wd"].notna().sum())

# Controle: komt het IMDb-id in Wikidata overeen met dat van TMDb?
check = df_api[["tmdb_id", "imdb_id"]].merge(df_db[["tmdb_id", "imdb_id_wd"]], on="tmdb_id")
print("IMDb-id's gelijk:", (check["imdb_id"] == check["imdb_id_wd"]).sum(), "/", len(check))

bewaar_proef("database_wikidata", df_db)
df_db

films uit de steekproef gevonden in Wikidata: 10 / 10; ontbrekend: geen
budget (USD) bekend    : 2
box office (USD) bekend: 3
IMDb-id's gelijk: 10 / 10
proef bewaard: data\proef\database_wikidata.json


,tmdb_id,wikidata_id,imdb_id_wd,budget_usd_wd,box_office_usd_wd,n_stellingen
0,101267,Q1199262,tt2215719,13000000.0,32726956.0,1
1,11761,Q2581569,tt1188113,NaN,NaN,1
2,127372,Q3724802,tt2103264,NaN,NaN,1
3,149786,Q18703100,tt2355762,NaN,NaN,1
4,296867,Q24034460,tt4457020,NaN,2540144.0,1
5,345925,Q28540702,tt4578118,NaN,NaN,1
6,348320,Q22443136,tt4849176,NaN,NaN,1
7,51209,Q1004440,tt1416801,NaN,NaN,1
8,56780,Q519619,tt0480239,20000000.0,4627375.0,1
9,84111,Q710908,tt2036416,NaN,NaN,1


In [12]:
# Hoeveel records zijn er in totaal? Eén telvraag aan de bron zelf.
telling = sparql("""
SELECT (COUNT(DISTINCT ?film) AS ?met_tmdb)
       (COUNT(DISTINCT ?filmBo) AS ?met_box_office)
       (COUNT(DISTINCT ?filmBoBudget) AS ?met_beide) WHERE {
  ?film wdt:P31 wd:Q11424 ; wdt:P4947 [] ; wdt:P577 ?datum .
  FILTER(YEAR(?datum) >= 2010 && YEAR(?datum) <= 2019)
  OPTIONAL { ?film wdt:P2142 [] . BIND(?film AS ?filmBo)
             OPTIONAL { ?film wdt:P2130 [] . BIND(?film AS ?filmBoBudget) } }
}""", timeout=90)
print("Wikidata-films 2010–2019 met TMDb-id (alle, ook zonder stemmen-filter):")
print(telling.T.to_string(header=False))

# Randgeval: een kapotte query geeft een 400 met een leesbare foutmelding.
try:
    sparql("SELECT ?x WHERE { dit is geen sparql }")
except RuntimeError as fout:
    print("\nrandgeval kapotte query ->", str(fout)[:120])

Wikidata-films 2010–2019 met TMDb-id (alle, ook zonder stemmen-filter):
met_tmdb        49167
met_box_office   1726
met_beide        1134



randgeval kapotte query -> status 400: SPARQL-QUERY: queryStr=SELECT ?x WHERE { dit is geen sparql }
java.util.concurrent.ExecutionException: org.o


**Vul in na de proef:**

- [x] Tien records opgehaald — alle **10/10** films gevonden via `wdt:P4947` (TMDb-id); de IMDb-id's komen 10/10 overeen met die van TMDb
- [x] De velden die ik nodig heb zitten erin: budget in USD 2/10, box office in USD 3/10. In deze steekproef
  had elke film hooguit één stelling. De aggregatie en `validate="one_to_one"` vangen het geval met meer stellingen af.
- [x] Totaal beschikbaar aantal records: **49.167** films (2010–2019) met TMDb-id, waarvan **1.726** met box office
  en **1.134** met budget én box office (`COUNT` op de endpoint zelf).
- [x] `robots.txt` / voorwaarden nagekeken: gestructureerde data valt onder **CC0 1.0**
  ([Wikidata:Licensing](https://www.wikidata.org/wiki/Wikidata:Licensing), gelezen 23-09-2026). De query service
  heeft een tijdslimiet van 60 s per query, en het Wikimedia-User-Agent-beleid geldt ook hier.
- [x] Persoonsgegevens in deze bron: nee (we vragen alleen film-items op).

*Conclusie:* **Go** als databasebron en als **koppeltabel**: TMDb-id ↔ IMDb-id ↔ Wikipedia-artikel.
Als succesmaat is Wikidata te dun (1.134 films met beide bedragen), maar het vult TMDb aan.
Randgeval: een kapotte query geeft status 400 met een leesbare Java-foutmelding, en die
laten we zien in plaats van een lege tabel.

---

# Deel 4 — Je beeldbron

### Bron 4 — TMDb-posters

| | |
|---|---|
| **Bron** | `https://image.tmdb.org/t/p/w342{poster_path}` (TMDb-beeldserver; `poster_path` komt uit bron 1) |
| **Deelvraag** | DV4: welke visuele kenmerken hangen samen met filmsucces? |
| **Techniek** | API (bestanden downloaden) |
| **Modaliteit** | beeld |
| **Sleutel nodig?** | nee |

**Wat je hier moet aantonen:** tien echte records, en dat de velden erin zitten die je
voor je deelvraag nodig hebt.

> In het PvA stond OMDb als beeldbron. We gebruiken de TMDb-poster, omdat TMDb vaste formaten
> heeft (`w342` = 342 px breed) en `poster_path` al in bron 1 zit. Dan zijn alle posters even groot.

Voor beeld toon je twee dingen aan:

1. je kunt de afbeeldingen **ophalen en opslaan**;
2. het is **zinvol** om er features uit te halen.

Punt 2 is waar het meestal misgaat. Beantwoord expliciet:

* Welke feature ga je uit deze beelden halen? *(afmetingen, dominante kleuren,
  helderheid, aanwezigheid van objecten, embeddings van een voorgetraind model, …)*
* Waarom helpt die feature je onderzoeksvraag beantwoorden?
* Zijn de beelden onderling vergelijkbaar? *(Verschillende formaten en belichting zijn
  op te lossen; totaal verschillende soorten beelden niet.)*

In [13]:
from PIL import Image

beeldmap = PROEF / "beeld"
beeldmap.mkdir(exist_ok=True)

KLEURNAMEN = ["rood", "oranje", "geel", "geelgroen", "groen", "groen-cyaan",
              "cyaan", "azuur", "blauw", "violet", "magenta", "roze"]


def beeldfeatures(pad: Path) -> dict:
    """Eenvoudige, uitlegbare beeldfeatures van een poster."""
    img = Image.open(pad).convert("RGB")
    rgb = np.asarray(img, dtype=float)
    grijs = np.asarray(img.convert("L"), dtype=float) / 255
    hsv = np.asarray(img.convert("HSV"), dtype=float)
    r, g, b = rgb[..., 0], rgb[..., 1], rgb[..., 2]
    rg, yb = r - g, 0.5 * (r + g) - b          # Hasler & Süsstrunk (2003)
    kleurrijkheid = np.hypot(rg.std(), yb.std()) + 0.3 * np.hypot(rg.mean(), yb.mean())
    # dominante tint: alleen pixels die echt gekleurd zijn (verzadiging > 25%)
    gekleurd = hsv[..., 1] > 0.25 * 255
    tinten = hsv[..., 0][gekleurd]
    dominant = (KLEURNAMEN[np.bincount((tinten / 256 * 12).astype(int), minlength=12).argmax()]
                if tinten.size > 0.05 * grijs.size else "grijs/zwart-wit")
    return {
        "breedte": img.width, "hoogte": img.height,
        "helderheid": round(grijs.mean(), 3),
        "contrast": round(grijs.std(), 3),
        "verzadiging": round(hsv[..., 1].mean() / 255, 3),
        "kleurrijkheid": round(kleurrijkheid, 1),
        "dominante_tint": dominant,
    }


beeld_rijen = []
for film in df_api.itertuples():
    if not film.poster_path:
        beeld_rijen.append({"tmdb_id": film.tmdb_id, "poster_ok": False})
        continue
    antwoord = requests.get(f"https://image.tmdb.org/t/p/w342{film.poster_path}",
                            headers=HEADERS, timeout=TIMEOUT)
    ok = antwoord.status_code == 200 and antwoord.headers.get("Content-Type", "").startswith("image/")
    rij = {"tmdb_id": film.tmdb_id, "poster_ok": ok}
    if ok:
        pad = beeldmap / f"{film.tmdb_id}.jpg"
        pad.write_bytes(antwoord.content)
        rij |= {"bytes": len(antwoord.content)} | beeldfeatures(pad)
    beeld_rijen.append(rij)

df_beeld = pd.DataFrame(beeld_rijen)
vat_samen(df_beeld, "posters")
print("bestanden:", sorted(p.name for p in beeldmap.iterdir()))

# Randgeval: een poster-pad dat niet bestaat.
fout = requests.get("https://image.tmdb.org/t/p/w342/bestaat_niet.jpg", headers=HEADERS, timeout=TIMEOUT)
print("randgeval onbekend posterpad -> status", fout.status_code)

bewaar_proef("beeld_posters", df_beeld)
df_beeld

--- posters ---
vorm      : 10 rijen x 10 kolommen
kolommen  : ['tmdb_id', 'poster_ok', 'bytes', 'breedte', 'hoogte', 'helderheid', 'contrast', 'verzadiging', 'kleurrijkheid', 'dominante_tint']
lege cellen per kolom:
tmdb_id           0
poster_ok         0
bytes             0
breedte           0
hoogte            0
helderheid        0
contrast          0
verzadiging       0
kleurrijkheid     0
dominante_tint    0
bestanden: ['101267.jpg', '11761.jpg', '127372.jpg', '149786.jpg', '296867.jpg', '345925.jpg', '348320.jpg', '51209.jpg', '56780.jpg', '84111.jpg']


randgeval onbekend posterpad -> status 404
proef bewaard: data\proef\beeld_posters.json


,tmdb_id,poster_ok,bytes,breedte,hoogte,helderheid,contrast,verzadiging,kleurrijkheid,dominante_tint
0,348320,True,76985,342,513,0.588,0.234,0.579,131.4,rood
1,56780,True,54324,342,513,0.370,0.182,0.294,34.2,cyaan
2,11761,True,32158,342,513,0.216,0.207,0.776,133.6,rood
3,345925,True,26475,342,507,0.683,0.268,0.252,37.8,cyaan
4,149786,True,49588,342,513,0.449,0.310,0.538,65.2,rood
5,127372,True,46047,342,513,0.344,0.251,0.110,9.8,cyaan
6,101267,True,57891,342,513,0.441,0.257,0.586,88.3,rood
7,84111,True,31947,342,513,0.707,0.307,0.107,17.1,azuur
8,296867,True,62619,342,513,0.620,0.326,0.271,59.6,rood
9,51209,True,41541,342,513,0.722,0.319,0.145,33.2,rood


**Mijn voorgenomen feature(s):** helderheid (gemiddelde grijswaarde), contrast (spreiding van
de grijswaarde), verzadiging, kleurrijkheid (Hasler & Süsstrunk, 2003) en dominante tint.

**Waarom die de onderzoeksvraag helpen beantwoorden:** posters volgen genre- en
marketingconventies: donker en onverzadigd bij thriller/horror, fel en kleurrijk bij
familie/animatie. De vraag is of die visuele keuze met de opbrengst samenhangt **nadat we voor
genre corrigeren**. Anders meten we alleen genre opnieuw. De features zijn gemiddelden en
spreidingen per pixel, dus ze hangen niet af van het formaat. Alle posters zijn hetzelfde soort
beeld (een filmposter van 342 px breed), dus ze zijn onderling vergelijkbaar.


**Vul in na de proef:**

- [x] Tien records opgehaald — 10/10 posters gedownload (342 × 513 px, één keer 342 × 507), opgeslagen in `data/proef/beeld/`
- [x] De velden die ik nodig heb zitten erin: alle vijf de features zijn voor 10/10 berekend; content-type gecontroleerd
- [x] Totaal beschikbaar aantal records: `poster_path` was 10/10 gevuld, dus ≈ alle **12.084** films
- [x] `robots.txt` / voorwaarden nagekeken: posters zijn auteursrechtelijk beschermd (studio/distributeur) en
  vallen onder de [TMDb API Terms of Use](https://www.themoviedb.org/api-terms-of-use). We gebruiken ze alleen
  lokaal om features te berekenen; de afbeeldingen publiceren of committen we niet.
- [x] Persoonsgegevens in deze bron: posters tonen vaak gezichten van acteurs. We doen geen gezichtsherkenning
  en slaan niets per persoon op.

*Conclusie:* **Go** — de bron met de beste dekking. Randgeval: een onbekend posterpad geeft 404.
Omdat we op statuscode én content-type controleren, slaan we nooit een foutpagina op als `.jpg`.

---

# Deel 5 — Je audiobron

### Bron 5 — Deezer (30-seconden-previews van soundtracks)

| | |
|---|---|
| **Bron** | Deezer API: `GET https://api.deezer.com/search/album?q=…`, `/album/{id}`, `/album/{id}/tracks`, veld `preview` (mp3, 30 s) |
| **Deelvraag** | DV5: welke rol spelen audio-eigenschappen van soundtracks? |
| **Techniek** | API |
| **Modaliteit** | audio |
| **Sleutel nodig?** | nee |

**Wat je hier moet aantonen:** tien echte records, en dat de velden erin zitten die je
voor je deelvraag nodig hebt.

> Dit is de bron waarop de meeste plannen een **no-go** krijgen — niet omdat audio
> moeilijk is, maar omdat het er vaak met de haren bij gesleept wordt.

Toon aan:

1. je kunt de audiobestanden **ophalen** (en mag dat: let op auteursrecht bij muziek);
2. er komt een **zinvolle feature** uit die met je vraag te maken heeft.

Voorbeelden van features die werken: spreektempo, toonhoogte, geluidsniveau,
spectrale features (MFCC), aanwezigheid van bepaalde geluiden, of tekst uit
spraakherkenning. Week 13 gaat hierover.

**Rode vlaggen:**

* je neemt zelf iets op — dat is data *produceren*, geen data *verzamelen*;
* de audio staat inhoudelijk los van je vraag ("willekeurige podcasts over het
  onderwerp");
* je hebt maar drie fragmenten.

In [14]:
import soundfile as sf   # pip install soundfile  (leest mp3 zonder ffmpeg)

audiomap = PROEF / "audio"
audiomap.mkdir(exist_ok=True)


def normaliseer(tekst: str) -> str:
    return re.sub(r"[^a-z0-9 ]", "", tekst.lower())


SOUNDTRACKWOORDEN = ["soundtrack", "motion picture", "original score", "music from"]


def zoek_soundtrack(titel: str, jaar: int):
    """Zoek op Deezer het soundtrackalbum van een film. Een treffer moet (1) met de
    filmtitel beginnen, (2) naar een soundtrack/score verwijzen en (3) binnen een jaar
    van de film zijn uitgebracht. Geeft (album, aantal afgewezen kandidaten)."""
    # Twee zoekvragen: met 'soundtrack' mist Deezer soms 'Music from the Motion Picture'.
    kandidaten = {}
    for q in [titel, f"{titel} soundtrack"]:
        d = haal_json("https://api.deezer.com/search/album", {"q": q})
        if "error" in d:                  # Deezer meldt fouten met status 200
            raise RuntimeError(d["error"])
        for album in d.get("data", [])[:10]:
            kandidaten.setdefault(album["id"], album)
    afgewezen = 0
    for album in kandidaten.values():
        at = normaliseer(album["title"])
        if not any(w in at for w in SOUNDTRACKWOORDEN):
            continue
        if not at.startswith(normaliseer(titel)):
            afgewezen += 1                # bv. 'Poker Face … (Peacock Original Series)'
            continue
        details = haal_json(f"https://api.deezer.com/album/{album['id']}")
        if abs(int(details.get("release_date", "0")[:4]) - jaar) > 1:
            afgewezen += 1                # zelfde titel, andere film of heruitgave
            continue
        return album, afgewezen
    return None, afgewezen


def audiofeatures(pad: Path) -> dict:
    """Duur, samplerate en een paar spectrale features, alleen met numpy."""
    y, sr = sf.read(pad, always_2d=True)
    y = y.mean(axis=1)                                    # stereo -> mono
    frame, hop = 2048, 1024
    frames = np.lib.stride_tricks.sliding_window_view(y, frame)[::hop]
    rms = np.sqrt((frames ** 2).mean(axis=1)) + 1e-10
    spec = np.abs(np.fft.rfft(frames * np.hanning(frame), axis=1))
    freqs = np.fft.rfftfreq(frame, 1 / sr)
    centroid = (spec * freqs).sum(axis=1) / (spec.sum(axis=1) + 1e-10)
    # ruwe tempo-schatting: autocorrelatie van de 'spectral flux'
    flux = np.maximum(np.diff(spec, axis=0), 0).sum(axis=1)
    flux = flux - flux.mean()
    ac = np.correlate(flux, flux, mode="full")[len(flux) - 1:]
    lags = np.arange(len(ac)) * hop / sr                  # seconden
    bereik = (lags >= 60 / 180) & (lags <= 60 / 60)       # 60–180 bpm
    tempo = 60 / lags[bereik][ac[bereik].argmax()]
    return {
        "duur_s": round(len(y) / sr, 1), "samplerate": sr,
        "luidheid_db": round(20 * np.log10(rms).mean(), 1),
        "dynamiek_db": round((20 * np.log10(rms)).std(), 1),
        "spectral_centroid_hz": round(centroid.mean()),
        "tempo_bpm_schatting": round(tempo),
    }


audio_rijen = []
for film in df_api.itertuples():
    album, afgewezen = zoek_soundtrack(film.titel, film.jaar)
    rij = {"tmdb_id": film.tmdb_id, "titel": film.titel,
           "album_gevonden": album is not None, "afgewezen_kandidaten": afgewezen}
    if album:
        tracks = haal_json(f"https://api.deezer.com/album/{album['id']}/tracks")["data"]
        track = next((t for t in tracks if t.get("preview")), None)
        rij |= {"album": album["title"], "n_tracks": len(tracks)}
        if track:
            pad = audiomap / f"{film.tmdb_id}.mp3"
            pad.write_bytes(requests.get(track["preview"], headers=HEADERS, timeout=TIMEOUT).content)
            info = haal_json(f"https://api.deezer.com/track/{track['id']}")
            rij |= {"track": track["title"], "deezer_bpm": info.get("bpm"),
                    "deezer_gain": info.get("gain"), "bytes": pad.stat().st_size}
            rij |= audiofeatures(pad)
    audio_rijen.append(rij)
    time.sleep(0.2)                                       # Deezer: max. 50 verzoeken / 5 s

df_audio = pd.DataFrame(audio_rijen)
vat_samen(df_audio, "Deezer")
print("soundtrack gevonden:", df_audio["album_gevonden"].sum(), "/ 10",
      "| kandidaten afgewezen als verkeerde film:", df_audio["afgewezen_kandidaten"].sum())

# Randgeval: Deezer geeft bij een onbekend album óók status 200, met een 'error'-veld.
print("randgeval /album/0 ->", haal_json("https://api.deezer.com/album/0"))

bewaar_proef("audio_deezer", df_audio)
df_audio

--- Deezer ---
vorm      : 10 rijen x 16 kolommen
kolommen  : ['tmdb_id', 'titel', 'album_gevonden', 'afgewezen_kandidaten', 'album', 'n_tracks', 'track', 'deezer_bpm', 'deezer_gain', 'bytes', 'duur_s', 'samplerate', 'luidheid_db', 'dynamiek_db', 'spectral_centroid_hz', 'tempo_bpm_schatting']
lege cellen per kolom:
tmdb_id                 0
titel                   0
album_gevonden          0
afgewezen_kandidaten    0
album                   9
n_tracks                9
track                   9
deezer_bpm              9
deezer_gain             9
bytes                   9
duur_s                  9
samplerate              9
luidheid_db             9
dynamiek_db             9
spectral_centroid_hz    9
tempo_bpm_schatting     9
soundtrack gevonden: 1 / 10 | kandidaten afgewezen als verkeerde film: 25


randgeval /album/0 -> {'error': {'type': 'DataException', 'message': 'no data', 'code': 800}}
proef bewaard: data\proef\audio_deezer.json


,tmdb_id,titel,album_gevonden,afgewezen_kandidaten,album,n_tracks,track,deezer_bpm,deezer_gain,bytes,duur_s,samplerate,luidheid_db,dynamiek_db,spectral_centroid_hz,tempo_bpm_schatting
0,348320,Scream Week,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,56780,Atlas Shrugged: Part I,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,11761,Peacock,False,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,345925,Rupture,False,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,149786,Christmas with Holly,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,127372,Emperor,True,7,Emperor (Original Motion Picture Soundtrack),17.0,Opening Title,0.0,-17.7,479827.0,30.0,44100.0,-28.3,6.7,1110.0,161.0
6,101267,Katy Perry: Part of Me,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,84111,You Are the Apple of My Eye,False,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,296867,El Jeremías,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,51209,Kill the Irishman,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Mijn voorgenomen feature(s):** gemiddelde luidheid (RMS in dB), dynamiek (spreiding van de
luidheid), spectral centroid (hoe 'helder' de klank is), een ruwe tempo-schatting en Deezers eigen `gain`.
Het veld `bpm` van Deezer staat in de proef steeds op `0`, dus tempo moeten we zelf berekenen.

**Waarom die de onderzoeksvraag helpen beantwoorden:** een luide, snelle en heldere soundtrack
past bij actie- en blockbusterfilms, een zachte en donkere bij drama. De vraag is of dat verband
houdt met succes, los van genre. Eerlijk gezegd is dit de zwakste schakel: 30 seconden van één
nummer vertegenwoordigt niet de hele film.

**Auteursrecht / licentie van deze audio:** de muziek is auteursrechtelijk beschermd (labels en
componisten). De [Deezer API Terms of Use](https://developers.deezer.com/termsofuse) (gelezen 23-09-2026)
beperken gebruik tot **strikt niet-commercieel** (art. IV) en verbieden om maatregelen te omzeilen
*"for the purpose of downloading the Content"* (art. V). Het preview-bestand opslaan is daarom
een grijs gebied. Wij gebruiken het alleen voor feature-extractie, committen het niet (staat in
`.gitignore`) en leggen dit in het go/no-go-gesprek aan de docent voor.


**Vul in na de proef:**

- [x] Tien records opgehaald — 10 zoekopdrachten uitgevoerd, maar slechts **1/10** correcte soundtrack
  gevonden (*Emperor*, 30,0 s, 44,1 kHz, ≈ 480 kB). **25 kandidaten afgewezen** omdat ze bij een andere
  titel hoorden, bv. *"Poker Face: Season 1 (Peacock Original Series)"* bij de film *Peacock*. Onze eerste,
  lossere matching telde die nog als treffer (4/10); de strengere check (titelbegin + jaar ± 1) haalt ze eruit.
- [x] De velden die ik nodig heb zitten erin: preview-url, duur, samplerate en de zelf berekende features.
  Deezers `bpm` is 0 en dus onbruikbaar.
- [x] Totaal beschikbaar aantal records: Deezer heeft geen telvraag per film. Met 1/10 dekking schatten
  we ≈ 1.200 films; bij bekende films (≥ 1000 stemmen) was het 5/10, dus ≈ 900 van de 1.843.
- [x] `robots.txt` / voorwaarden nagekeken: zie auteursrecht hierboven.
- [x] Persoonsgegevens in deze bron: nee (artiestnamen zijn publieke makers; we slaan alleen album- en tracktitel op).

*Conclusie:* technisch **go**: previews zijn op te halen en te decoderen (met `soundfile`, zonder ffmpeg).
Inhoudelijk **go onder voorwaarde**: bij willekeurige films is de dekking 1/10, en dat is te weinig.
Alleen met de afbakening op bekende films (5/10) en een strikte titel-en-jaar-match is audio haalbaar.
Rode vlaggen: niet zelf opgenomen ✔; aantal (≈ 900) ✔; inhoudelijke band met de vraag ⚠ (zwak).

---

# Deel 6 — De haalbaarheidstabel

Dit is de tabel die je in het go/no-go-gesprek laat zien. Vul hem eerlijk in: een
bron waarvan je nu weet dat hij niet werkt, is winst. *(Proef gedraaid op 23-09-2026; de cel
hieronder rekent de aantallen opnieuw uit.)*

| # | Bron | Techniek | Modaliteit | Gelukt? | Records in de proef | Totaal beschikbaar | Blokkade / vervolgstap |
|---|---|---|---|---|---|---|---|
| 1 | TMDb API (+ OMDb) | API | tabel + tekst | ja | 10 (+ 10 OMDb) | 12.084 (≥ 50 stemmen); 1.843 (≥ 1000) | budget 2/10 en reviews 3/10 bij willekeurige films → afbakening naar bekende films |
| 2 | Wikipedia-infobox | scraping | tabel (HTML) | ja | 9 | ≈ 90 % van de films | **geen login** → eis "scraping met login" niet gedekt; met docent afstemmen |
| 3 | Wikidata SPARQL | database | tabel | ja | 10 | 49.167 met TMDb-id; 1.134 met budget + box office | alleen als koppel- en aanvulbron, niet als enige succesmaat |
| 4 | TMDb-posters | API (bestanden) | beeld | ja | 10 | ≈ 12.084 | geen |
| 5 | Deezer-previews | API | audio | deels | 1 (van 10 zoekvragen) | ≈ 1.200; ≈ 900 bij bekende films | dekking; strikte titel+jaar-match; opslaan previews voorleggen (Deezer ToS art. V) |

**Controle op de eisen van de eindopdracht:**

| Eis | Gedekt door bron # | Aangetoond? |
|---|---|---|
| Webscraping (met login op de bron) | 2 | **deels** — scraping ja, login nee (Letterboxd verbiedt scrapers in de ToS) |
| API | 1, 4, 5 | ja |
| Publieke database | 3 | ja (SPARQL-query op Wikidata) |
| Ongestructureerde tekst (zinnen) | 1 (TMDb-reviews) | ja, maar dun: 3/10 (8/10 bij bekende films) |
| Beeld | 4 | ja (10/10) |
| Audio | 5 | deels (1/10; 5/10 bij bekende films) |

**Mijn grootste risico:** de eindtabel. Bij een inner join over opbrengst, budget, beeld en audio
blijft er van de tien films **niets** over (0/10). Audio (1/10) en budget (4/10) zijn de knelpunten,
en ontbrekende waarden hangen samen met hoe bekend een film is (zie de uitdaging).

**Plan B daarvoor:** (1) de eenheid van analyse aanpassen naar films uit 2010–2019 met **≥ 1000
stemmen** (1.843 films; 5/10 compleet → ≈ 900 rijen); (2) audio als **left join** meenemen, met een
aparte subanalyse op de films met soundtrack; (3) voor "scraping met login" met de docent bespreken
welke bron met login wél mag, want Letterboxd valt af.

In [15]:
# De getallen voor de haalbaarheidstabel, rechtstreeks uit de proef.
overzicht = pd.DataFrame([
    ("TMDb API (+ reviews)", "API", "tabel + tekst", len(df_api), TOTAAL_TMDB),
    ("OMDb API", "API", "tabel", int(df_omdb["omdb_gevonden"].sum()), "per imdb_id"),
    ("Wikipedia-infobox", "scraping", "tabel (HTML)", int(df_wiki["wiki_url"].notna().sum()), "zie Wikidata-telling"),
    ("Wikidata SPARQL", "database", "tabel", len(df_db), int(telling.loc[0, "met_tmdb"])),
    ("TMDb-posters", "API (bestand)", "beeld", int(df_beeld["poster_ok"].sum()), TOTAAL_TMDB),
    ("Deezer previews", "API", "audio", int(df_audio["album_gevonden"].sum()), "≈ dekking × totaal"),
], columns=["bron", "techniek", "modaliteit", "records_in_proef", "totaal_beschikbaar"])
overzicht

,bron,techniek,modaliteit,records_in_proef,totaal_beschikbaar
0,TMDb API (+ reviews),API,tabel + tekst,10,12084
1,OMDb API,API,tabel,10,per imdb_id
2,Wikipedia-infobox,scraping,tabel (HTML),9,zie Wikidata-telling
3,Wikidata SPARQL,database,tabel,10,49167
4,TMDb-posters,API (bestand),beeld,10,12084
5,Deezer previews,API,audio,1,≈ dekking × totaal


---

# ⚠️ Uitdaging — reken je eindtabel uit

Alle bronnen werkend? Zoek dan nu al uit hoeveel er van je eindtabel overblijft. Dit
is het probleem dat groepen in week 14 verrast, en het kost je nu een kwartier.

1. Zet per bron je tien proefrecords in een DataFrame.
2. Bepaal per bron de **granulariteit**: één rij is één …?
3. Bepaal de **joinsleutel**: op welk veld koppel je de bronnen?
4. Doe een `merge` op je proefdata en tel hoeveel rijen je overhoudt.
5. Schaal op: als je van bron A 380 records hebt en bron D er maar 45 dekt, hoe groot
   wordt je eindtabel dan?

| Bron | Granulariteit | Aantal records (verwacht, volledig) | Joinsleutel | Dekking (proef) |
|---|---|---|---|---|
| TMDb details | 1 film | 12.084 (≥ 50 st.) / 1.843 (≥ 1000 st.) | `tmdb_id` | 10/10 |
| TMDb reviews | 1 review → **aggregeren** per film | enkele per film | `tmdb_id` | 3/10 films met ≥ 1 review |
| OMDb | 1 film | 1 per `imdb_id` | `imdb_id` | 10/10 (box office 5/10) |
| Wikipedia | 1 pagina = 1 film | ≈ 90 % | `tmdb_id` (gezocht op titel + jaar, gecontroleerd op `imdb_id`) | 9/10 |
| Wikidata | 1 stelling (budget/box office per regio) → **aggregeren** per film | 49.167 | `tmdb_id` (P4947) | 10/10; bedragen 2–3/10 |
| Posters | 1 poster per film | ≈ 12.084 | `tmdb_id` | 10/10 |
| Deezer | 1 track (album ≈ 17 tracks) → 1 preview per film | ≈ 1.200 | titel + jaar (fuzzy!) → `tmdb_id` | 1/10 (5/10 bij bekende films) |

De cellen hieronder doen stap 1–5 met onze eigen proefdata.

In [16]:
# 1–4: proefdata samenvoegen op tmdb_id en tellen wat er overblijft.
basis = df_api[["tmdb_id", "imdb_id", "titel", "jaar", "budget_usd", "revenue_usd", "n_reviews"]]
delen = {
    "omdb":     df_omdb[["imdb_id", "box_office_us", "metascore"]],
    "wiki":     df_wiki[["tmdb_id", "budget_usd_wiki", "box_office_usd_wiki"]],
    "wikidata": df_db[["tmdb_id", "budget_usd_wd", "box_office_usd_wd"]],
    "beeld":    df_beeld.reindex(columns=["tmdb_id", "helderheid", "kleurrijkheid"]),
    "audio":    df_audio.reindex(columns=["tmdb_id", "luidheid_db", "spectral_centroid_hz"]),
}

eind = basis.copy()
for naam, deel in delen.items():
    sleutel = "imdb_id" if naam == "omdb" else "tmdb_id"
    # validate='one_to_one' faalt meteen als een bron meer dan één rij per film heeft
    eind = eind.merge(deel, on=sleutel, how="left", validate="one_to_one")
assert len(eind) == len(basis), "left join mag geen rijen toevoegen"

# Succesmaat: opbrengst uit TMDb, anders Wikidata, anders Wikipedia.
eind["opbrengst_usd"] = (eind["revenue_usd"].fillna(eind["box_office_usd_wd"])
                         .fillna(eind["box_office_usd_wiki"]))
eind["budget_best"] = (eind["budget_usd"].fillna(eind["budget_usd_wd"])
                       .fillna(eind["budget_usd_wiki"]))
dekking = pd.Series({
    "opbrengst (een van de bronnen)": eind["opbrengst_usd"].notna().mean(),
    "budget (een van de bronnen)":    eind["budget_best"].notna().mean(),
    "≥1 review":                      (eind["n_reviews"] > 0).mean(),
    "poster-features":                eind["helderheid"].notna().mean(),
    "audio-features":                 eind["luidheid_db"].notna().mean(),
})
compleet = eind[["opbrengst_usd", "budget_best", "helderheid", "luidheid_db"]].notna().all(axis=1)
eind = eind.assign(compleet=compleet)
bewaar_proef("eindtabel_proef", eind)
print("left join :", len(eind), "rijen")
print("inner join:", int(compleet.sum()), "rijen met opbrengst + budget + beeld + audio")
print("\ndekking per onderdeel in de steekproef:")
print(dekking.map("{:.0%}".format).to_string())
eind[["titel", "jaar", "opbrengst_usd", "budget_best", "n_reviews", "helderheid", "luidheid_db", "compleet"]]

proef bewaard: data\proef\eindtabel_proef.json
left join : 10 rijen
inner join: 0 rijen met opbrengst + budget + beeld + audio

dekking per onderdeel in de steekproef:
opbrengst (een van de bronnen)     70%
budget (een van de bronnen)        40%
≥1 review                          30%
poster-features                   100%
audio-features                     10%


,titel,jaar,opbrengst_usd,budget_best,n_reviews,helderheid,luidheid_db,compleet
0,Scream Week,2016,NaN,NaN,0,0.588,NaN,False
1,Atlas Shrugged: Part I,2011,4627375.0,20000000.0,0,0.370,NaN,False
2,Peacock,2010,NaN,NaN,0,0.216,NaN,False
3,Rupture,2016,31238.0,NaN,1,0.683,NaN,False
4,Christmas with Holly,2012,NaN,NaN,0,0.449,NaN,False
5,Emperor,2012,14858240.0,NaN,1,0.344,-28.3,False
6,Katy Perry: Part of Me,2012,32726956.0,12000000.0,0,0.441,NaN,False
7,You Are the Apple of My Eye,2011,24500000.0,1700000.0,0,0.707,NaN,False
8,El Jeremías,2016,2540144.0,NaN,0,0.620,NaN,False
9,Kill the Irishman,2011,1188194.0,12000000.0,1,0.722,NaN,False


In [17]:
# 5: opschalen naar de volledige afbakening.
print(f"TMDb-films 2010–2019 met ≥50 stemmen: {TOTAAL_TMDB}")
for onderdeel, fractie in dekking.items():
    print(f"  {onderdeel:32} ≈ {fractie * TOTAAL_TMDB:>6.0f} films")
print(f"  {'alles tegelijk (inner join)':32} ≈ {compleet.mean() * TOTAAL_TMDB:>6.0f} films")

# Onzekerheid: met n = 10 is een dekking van p erg onnauwkeurig (Wilson-interval, 95%).
from statistics import NormalDist
z = NormalDist().inv_cdf(0.975)
k, n = int(compleet.sum()), len(compleet)
p = k / n
midden = (p + z**2 / (2 * n)) / (1 + z**2 / n)
marge = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
print(f"\ninner-join-dekking {k}/{n}: 95%-interval {midden - marge:.0%} – {midden + marge:.0%}"
      f" -> {max(midden - marge, 0) * TOTAAL_TMDB:.0f} tot {(midden + marge) * TOTAAL_TMDB:.0f} films")

TMDb-films 2010–2019 met ≥50 stemmen: 12084
  opbrengst (een van de bronnen)   ≈   8459 films
  budget (een van de bronnen)      ≈   4834 films
  ≥1 review                        ≈   3625 films
  poster-features                  ≈  12084 films
  audio-features                   ≈   1208 films
  alles tegelijk (inner join)      ≈      0 films

inner-join-dekking 0/10: 95%-interval 0% – 28% -> 0 tot 3354 films


In [18]:
# Is de ontbrekende dekking willekeurig? Vergelijk films mét en zónder een onderdeel
# op een variabele die we wél voor alle tien hebben: vote_count (hoe bekend de film is).
vergelijk = df_api[["tmdb_id", "vote_count"]].merge(
    eind[["tmdb_id", "opbrengst_usd", "budget_best", "luidheid_db", "n_reviews"]], on="tmdb_id")
indicatoren = {
    "opbrengst bekend": vergelijk["opbrengst_usd"].notna(),
    "budget bekend":    vergelijk["budget_best"].notna(),
    "audio gevonden":   vergelijk["luidheid_db"].notna(),
    "≥1 review":        vergelijk["n_reviews"] > 0,
}
pd.DataFrame({
    naam: {"n ja": int(ind.sum()),
           "mediaan vote_count (ja)": vergelijk.loc[ind, "vote_count"].median(),
           "mediaan vote_count (nee)": vergelijk.loc[~ind, "vote_count"].median()}
    for naam, ind in indicatoren.items()
}).T

,n ja,mediaan vote_count (ja),mediaan vote_count (nee)
opbrengst bekend,7.0,290.0,100.0
budget bekend,4.0,266.0,181.5
audio gevonden,1.0,290.0,241.0
≥1 review,3.0,297.0,222.0


In [19]:
# Knop 'eenheid van analyse': wat als we alleen bekende films (≥ 1000 stemmen) nemen?
# Weer maximaal tien records, alleen de velden die de bottleneck vormen.
FILTER_B = FILTER | {"vote_count.gte": 1000}
eerste_b = tmdb("/discover/movie", **FILTER_B, sort_by=SORTERING, page=1)
TOTAAL_B = eerste_b["total_results"]

rng_b = random.Random(7)
paginas_b = rng_b.sample(range(1, eerste_b["total_pages"] + 1), 10)
b_rijen = []
for pagina in paginas_b:
    kandidaat = rng_b.choice(tmdb("/discover/movie", **FILTER_B, sort_by=SORTERING, page=pagina)["results"])
    f = tmdb(f"/movie/{kandidaat['id']}", append_to_response="reviews")
    album, _ = zoek_soundtrack(f["title"], int(f["release_date"][:4]))
    b_rijen.append({"titel": f["title"], "vote_count": f["vote_count"],
                    "budget": bool(f["budget"]), "revenue": bool(f["revenue"]),
                    "review": f["reviews"]["total_results"] > 0, "soundtrack": album is not None})
    time.sleep(0.25)
df_b = pd.DataFrame(b_rijen)
bewaar_proef("planb_bekende_films", df_b)
print(f"films 2010–2019 met ≥1000 stemmen: {TOTAAL_B} (was {TOTAAL_TMDB} bij ≥50)")
print(df_b[["budget", "revenue", "review", "soundtrack"]].mean().map("{:.0%}".format).to_string())
alles_b = df_b[["budget", "revenue", "soundtrack"]].all(axis=1)
print(f"budget + revenue + soundtrack: {alles_b.sum()}/10 -> ≈ {alles_b.mean() * TOTAAL_B:.0f} films")
df_b

proef bewaard: data\proef\planb_bekende_films.json
films 2010–2019 met ≥1000 stemmen: 1843 (was 12084 bij ≥50)
budget        70%
revenue       80%
review        80%
soundtrack    50%
budget + revenue + soundtrack: 5/10 -> ≈ 922 films


,titel,vote_count,budget,revenue,review,soundtrack
0,The Loft,1606,True,True,True,False
1,Cosmopolis,1382,True,True,True,False
2,Pan,2886,True,True,True,True
3,Polaroid,1164,False,False,True,False
4,16 Wishes,1382,False,False,False,False
5,Battle: Los Angeles,3849,True,True,True,True
6,God's Own Country,1012,False,True,True,False
7,From Up on Poppy Hill,1998,True,True,False,True
8,Tomorrowland,7509,True,True,True,True
9,Annihilation,9641,True,True,True,True


In [20]:
# ⚠️ De rekensom met onze eigen aantallen (plan B: bekende films, ≥ 1000 stemmen)
bron_a_rijen = TOTAAL_B                              # films in de afbakening
bron_d_dekking = round(alles_b.mean() * TOTAAL_B)    # met budget + revenue + soundtrack

print(f"inner join  -> ongeveer {min(bron_a_rijen, bron_d_dekking)} rijen")
print(f"left join   -> {bron_a_rijen} rijen, waarvan "
      f"{bron_a_rijen - bron_d_dekking} zonder de features uit bron D "
      f"({(bron_a_rijen - bron_d_dekking) / bron_a_rijen:.0%} ontbrekend)")

inner join  -> ongeveer 922 rijen
left join   -> 1843 rijen, waarvan 921 zonder de features uit bron D (50% ontbrekend)


**Verwachte omvang eindtabel:** met de oorspronkelijke afbakening (≥ 50 stemmen) is de
puntschatting voor een inner join **0 films**. Het 95 %-interval loopt tot ≈ 3.350, maar met 0 van
de 10 is dat geen basis voor een plan. Met de afbakening ≥ 1000 stemmen: 5/10 compleet
(budget + opbrengst + soundtrack) → **≈ 920 films** (95 %-interval 24–76 % → ≈ 440 tot 1.400).

**Is dat genoeg voor mijn vraag?** Met de oorspronkelijke afbakening niet. We passen de **eenheid
van analyse** aan (bekende films, ≥ 1000 stemmen) en gebruiken voor audio een **left join** met een
aparte subanalyse. ≈ 920 rijen is ruim genoeg om een correlatie van r = 0,2 aan te tonen, want
daarvoor zijn 194 films nodig (zie V1). De prijs: onze conclusies gelden dan alleen voor
bekende films, niet voor alle films. De bias-check laat zien dat films met een bekende opbrengst
een mediaan van 290 stemmen hebben tegen 100 zonder. Wat ontbreekt, ontbreekt dus niet willekeurig.

---

# Verdieping — experimentontwerp en causale DAG
### V1 — Power vóór data
Leg alpha, gewenste power, kleinste relevante effect en analyse vast. Bereken voor twee gelijke
groepen de benodigde n bij effectgroottes 0,5 en 0,25; verklaar de kwadratische groei.

In [21]:
# V1 — power-analyse vóór dataverzameling (normale benadering, stdlib).
from statistics import NormalDist

ALPHA, POWER = 0.05, 0.80
z_a = NormalDist().inv_cdf(1 - ALPHA / 2)
z_b = NormalDist().inv_cdf(POWER)


def n_per_groep(d: float) -> int:
    """Twee gelijke groepen, tweezijdige t-toets, effectgrootte Cohen's d."""
    return int(np.ceil(2 * (z_a + z_b) ** 2 / d ** 2))


def n_correlatie(r: float) -> int:
    """Totale n om een correlatie r ≠ 0 aan te tonen (Fisher-z)."""
    return int(np.ceil(((z_a + z_b) / np.arctanh(r)) ** 2 + 3))


for d in [0.5, 0.25]:
    print(f"d = {d:<4}: {n_per_groep(d):>4} films per groep, {2 * n_per_groep(d):>4} totaal")
print(f"verhouding: {n_per_groep(0.25) / n_per_groep(0.5):.1f}x  (halve d -> 4x zoveel n, want n ∝ 1/d²)")
print()
for r in [0.3, 0.2, 0.1]:
    print(f"correlatie r = {r}: n = {n_correlatie(r)} films")

d = 0.5 :   63 films per groep,  126 totaal
d = 0.25:  252 films per groep,  504 totaal
verhouding: 4.0x  (halve d -> 4x zoveel n, want n ∝ 1/d²)

correlatie r = 0.3: n = 85 films
correlatie r = 0.2: n = 194 films
correlatie r = 0.1: n = 783 films


**Vooraf vastgelegd:** α = 0,05 (tweezijdig), power = 0,80. Kleinste relevante effect:
**r = 0,2** tussen een beeld-/audiofeature en log(opbrengst). Bij een groepsvergelijking (bv. lichte
vs. donkere posters) is dat **d = 0,25**. Analyse: regressie van log(opbrengst) op de feature, met
genre, jaar en franchise als controlevariabelen (zie V2).

**Uitkomst:** d = 0,5 vraagt 63 films per groep, d = 0,25 vraagt 252, dus **4× zoveel**. De groei is
kwadratisch omdat de standaardfout van een verschil daalt met 1/√n. Om een half zo groot effect met
dezelfde zekerheid te zien, moet de standaardfout ook half zo groot worden, en dat kost 2² = 4× zoveel
data (n ∝ 1/d²). Voor ons: de ≈ 920 complete films uit plan B zijn genoeg voor r = 0,2 (194 nodig) en
voor d = 0,25 (504 totaal), maar niet voor r = 0,1 (783 nodig, en dan zou alles mee moeten zitten).

### V2 — Backdoorpaden
Formuleer behandeling, uitkomst, confounder, mediator en collider voor je PvA. Teken een DAG,
wijs backdoorpaden en minimale correctieset aan en noteer één niet-toetsbare causale aanname.

**Onze DAG** (behandeling = budget, uitkomst = wereldwijde opbrengst):

```
          ┌──────── genre · jaar · franchise ────────┐     confounders
          ▼                                          ▼
       budget ─────────────────────────────────► opbrengst
          │                                          ▲  │
          ├──► release_breedte ──────────────────────┘  │     mediator (bioscopen/marketing)
          │                                             │
          └─────────────► in_bron ◄─────────────────────┘     collider (staat met bedragen in bron)
```

| Rol | Variabele | Waarom |
|---|---|---|
| Behandeling | budget | wat een producent zelf kiest |
| Uitkomst | opbrengst (log, USD) | financieel succes uit de hoofdvraag |
| Confounders | genre, releasejaar, franchise/sequel | bepalen zowel het budget als de opbrengst (actie is duur én trekt publiek; inflatie; vervolgfilms krijgen meer geld én hebben een vast publiek) |
| Mediator | release_breedte (aantal bioscopen/marketing) | een groter budget koopt een bredere release, en die levert opbrengst op |
| Collider | in_bron (film heeft bedragen in Wikidata/Deezer-soundtrack) | grote producties én hits worden vaker gedocumenteerd. Een inner join selecteert hierop |

De code hieronder zoekt de backdoorpaden en controleert welke correctieset ze dicht zet.

In [22]:
# V2 — DAG voor 'budget -> opbrengst' en een automatische zoektocht naar backdoorpaden.
PIJLEN = [
    ("genre", "budget"), ("genre", "opbrengst"),
    ("jaar", "budget"), ("jaar", "opbrengst"),
    ("franchise", "budget"), ("franchise", "opbrengst"),
    ("budget", "release_breedte"), ("release_breedte", "opbrengst"),    # mediator
    ("budget", "opbrengst"),
    ("budget", "in_bron"), ("opbrengst", "in_bron"),                   # collider (selectie)
]
BEHANDELING, UITKOMST = "budget", "opbrengst"


def alle_paden(a, b, pad=None):
    """Alle paden zonder herhaling tussen a en b, pijlrichting genegeerd."""
    pad = pad or [a]
    if a == b:
        yield pad
        return
    for x, y in PIJLEN:
        for van, naar in [(x, y), (y, x)]:
            if van == a and naar not in pad:
                yield from alle_paden(naar, b, pad + [naar])


def is_collider(pad, i):
    return (pad[i - 1], pad[i]) in PIJLEN and (pad[i + 1], pad[i]) in PIJLEN


def geblokkeerd(pad, z):
    """Pad is dicht als een niet-collider in z zit, of een collider níet in z zit."""
    for i in range(1, len(pad) - 1):
        if is_collider(pad, i) != (pad[i] in z):
            return True
    return False


backdoor = [p for p in alle_paden(BEHANDELING, UITKOMST) if (p[1], p[0]) in PIJLEN]
print("backdoorpaden:")
for p in backdoor:
    print("  ", " — ".join(p))

for z in [set(), {"genre", "jaar", "franchise"},
          {"genre", "jaar", "franchise", "in_bron"},
          {"genre", "jaar", "franchise", "release_breedte"}]:
    open_paden = [p for p in alle_paden(BEHANDELING, UITKOMST)
                  if len(p) > 2 and not geblokkeerd(p, z)]
    print(f"\ncorrectieset {sorted(z) or '{}'}: open niet-directe paden:")
    for p in open_paden:
        soort = "backdoor" if (p[1], p[0]) in PIJLEN else "causaal/selectie"
        print(f"   [{soort}] ", " — ".join(p))

backdoorpaden:
   budget — genre — opbrengst
   budget — jaar — opbrengst
   budget — franchise — opbrengst

correctieset {}: open niet-directe paden:
   [backdoor]  budget — genre — opbrengst
   [backdoor]  budget — jaar — opbrengst
   [backdoor]  budget — franchise — opbrengst
   [causaal/selectie]  budget — release_breedte — opbrengst

correctieset ['franchise', 'genre', 'jaar']: open niet-directe paden:
   [causaal/selectie]  budget — release_breedte — opbrengst

correctieset ['franchise', 'genre', 'in_bron', 'jaar']: open niet-directe paden:
   [causaal/selectie]  budget — release_breedte — opbrengst
   [causaal/selectie]  budget — in_bron — opbrengst

correctieset ['franchise', 'genre', 'jaar', 'release_breedte']: open niet-directe paden:


**Backdoorpaden:** budget ← genre → opbrengst, budget ← jaar → opbrengst en
budget ← franchise → opbrengst. **Minimale correctieset: {genre, jaar, franchise}.** Daarmee zijn
alle backdoorpaden dicht. Alleen het causale pad via release_breedte blijft open, en dat hoort ook zo.

De code laat ook de twee fouten zien. Wie **conditioneert op in_bron** (en dat doet een inner join!),
opent het pad budget → in_bron ← opbrengst. Wie **corrigeert voor release_breedte**, blokkeert een
deel van het effect dat we willen meten.

**Niet-toetsbare causale aanname:** er is geen ongemeten confounder zoals *verwachte kwaliteit*.
Een studio geeft meer budget aan projecten waarvan ze succes verwacht (sterk script, bekende
regisseur), en die kwaliteit drijft óók de opbrengst. Met onze data kunnen we dat niet toetsen.
Daarom spreken we in het eindrapport van **samenhang**, niet van oorzaak.

---

## Theorievragen

De haalbaarheidscheck is een **methode**, geen invuloefening. Deze vragen gaan over de
redenering eronder: wat bewijst een proef van tien records eigenlijk, en wat niet? Ze
hebben geen modelantwoord — het antwoord hangt van jullie bronnen af — maar wél een
acceptatiecriterium. Beantwoord ze als groep in de Markdown-cel hieronder; in het
go/no-go-gesprek wordt hierop doorgevraagd.

### T1 — Wat bewijzen tien records, en wat niet?

1. Verdeel de volgende eigenschappen over drie bakjes: **aangetoond met tien records**,
   **aan te tonen met één telvraag aan de bron**, en **pas aan te tonen met de volledige
   extractie**. Motiveer per eigenschap.
   *Toegang · veldnamen · datatypes · totaal aantal records · dekking over de hele periode ·
   het aandeel lege waarden · of de joinsleutel uniek is · of de licentie het toestaat ·
   of er genoeg variatie in de doelvariabele zit.*
2. Je tien records zijn de eerste tien die de bron teruggaf. Waarom is dat géén
   aselecte steekproef, en welke drie sorteringen zorgen er in de praktijk voor dat je
   juist de makkelijkste records ziet? Wat zou je erbij ophalen om die vertekening te
   verkleinen zonder je limiet van tien records los te laten?
3. Bron 2 lukte niet binnen 30 minuten. Formuleer die uitkomst als een bruikbare
   conclusie in één zin: wat is er precies geprobeerd, waar liep het vast, en welk
   criterium is daarmee wel of niet afgevallen?

> **Je antwoord is af als** iemand die de bron niet kent, uit vraag 3 kan afleiden of hij
> het zelf nog eens zou moeten proberen.

### T2 — Dekking, joins en de vertekening die je erbij krijgt

1. Bron A heeft 380 records, bron D dekt er 45 van. Bereken de omvang van je eindtabel bij
   een inner join en bij een left join, en zeg per variant wat één rij dan voorstelt.
2. De left join houdt 380 rijen, dus "je verliest niets". Leg uit waarom dat argument in de
   analyse alsnog misgaat. Betrek in je antwoord het verschil tussen ontbrekende waarden
   die **willekeurig** verdeeld zijn en ontbrekende waarden die samenhangen met de bron
   zelf — bijvoorbeeld: alleen de populaire items hebben een review, alleen de recente
   dagen hebben weerdata.
3. Welke concrete controle op jullie proefdata laat zien of de ontbrekende dekking
   willekeurig is of systematisch? Beschrijf de vergelijking die je zou maken.
4. Bij welke van de twee joins is het eerlijk om de resultaten van je analyse te
   presenteren als geldig voor "alle items", en wat schrijf je op als dat niet mag?

> **Je antwoord is af als** er een getal in staat dat uit jullie eigen proefdata komt.

### T3 — Granulariteit: één rij is één …?

1. Schrijf per bron op wat één rij voorstelt, en welke combinatie van velden die rij
   **uniek** maakt. Toon met je proefdata aan dat die combinatie inderdaad uniek is; zo
   niet, dan heb je je granulariteit verkeerd benoemd.
2. Twee van jullie bronnen hebben vrijwel zeker een verschillende granulariteit (bijvoorbeeld één
   meting per uur tegenover één review per bezoek). Beschrijf welke van de twee je
   **aggregeert** en welke de granulariteit van je eindtabel bepaalt, en welk gemiddelde of welke
   som daarbij hoort.
3. Wat gebeurt er met je rijaantallen als je die aggregatie vergeet en meteen koppelt?
   Noem het verschijnsel bij naam en zeg welke controle het meteen zichtbaar maakt.
4. Beeld en audio hebben vaak hun eigen granulariteit: één foto per item, maar tien fragmenten per
   item. Hoe breng je die terug tot één rij, en welke informatie gooi je daarbij weg?

> **Je antwoord is af als** de zin "één rij in onze eindtabel is één …" precies één
> mogelijke lezing heeft.

### T4 — Is de eindtabel groot genoeg voor de vraag?

1. Stel dat er 45 rijen overblijven. Welke soorten uitspraken kun je daar nog mee
   onderbouwen, en welke beslist niet? Denk aan een verschil tussen groepen, een trend over
   de tijd en een voorspellend model.
2. Je maakt uit tekst, beeld en audio samen 60 features bij 45 rijen. Leg uit waarom dat
   modelmatig een probleem is, en waarom "meer features toevoegen" het erger maakt in
   plaats van beter.
3. Je hebt drie knoppen om aan te draaien: de **eenheid van analyse**, de **bron** en het
   **type join**. Beschrijf per knop welk effect draaien heeft op het aantal rijen én op de
   vraag die je nog kunt beantwoorden.
4. Welke van die drie kost het minste werk als je hem nú omzet, en welke het meeste als je
   hem pas in week 14 omzet?

> **Je antwoord is af als** je bij vraag 3 een keuze maakt en die met een getal onderbouwt.

### T5 — Een no-go is ook een resultaat

1. Leg uit wat een **spike** is en waarom er een harde tijdsgrens omheen staat. Wat is het
   verschil tussen "het is niet gelukt" en "het is niet haalbaar", en welk bewijs hoort bij
   de tweede uitspraak?
2. Jullie hebben al twee weken in een bron gestoken die nu blijkt te haperen. Benoem de
   denkfout die maakt dat groepen daar tóch mee doorgaan, en formuleer de vraag die het
   besluit weer op de toekomst richt.
3. Voor elke bron gelden drie soorten grenzen naast de technische: de
   **gebruiksvoorwaarden**, de **licentie** op hergebruik en publicatie, en de **AVG** als er
   persoonsgegevens in zitten. Noteer per bron welke van de drie speelt, en wat je concreet
   hebt gecontroleerd — niet "het is open data", maar welke versie van welke licentie.
4. Een bron werkt technisch maar mag niet. Waarom is dat voor je plan van aanpak precies
   even zwaar als een bron die niet werkt, en wat schrijf je in de kolom *blokkade*?

> **Je antwoord is af als** er per bron een controleerbare verwijzing staat: een URL naar de
> voorwaarden, de naam en versie van de licentie, of de datum waarop je het hebt nagevraagd.

### T6 — Power en causaliteit

1. Waarom is de kleinste relevante effectgrootte iets dat je vóór het verzamelen vastlegt, en
   niet iets dat je achteraf uit je eigen data afleest? Reken na wat er met de benodigde n
   gebeurt als je die grootte halveert.
2. Wanneer opent corrigeren voor een collider juist een pad, en wanneer blokkeert corrigeren
   voor een mediator een deel van het effect dat je wilde meten?

> **Je antwoord is af als** je bij vraag 2 je eigen DAG uit V2 aanwijst en per variabele zegt
> of je er wel of niet voor corrigeert, met de reden erbij.

*Jullie antwoorden:*

**T1. Wat bewijzen tien records?**

1. *Aangetoond met tien records:* **toegang** (10/10 status 200 bij alle bronnen), **veldnamen**
   (`budget`, `revenue`, `poster_path` …) en **datatypes**. Aan de types zagen we ook dat TMDb `0` gebruikt
   voor "onbekend" (8/10 bij budget). Daarvoor heb je alleen de vorm nodig, niet de volledige data.
   *Aan te tonen met één telvraag:* **totaal aantal records** (`total_results` = 12.084; Wikidata-`COUNT` =
   49.167 / 1.726 / 1.134), **dekking over de periode** (dezelfde telvraag per jaar) en **of de licentie
   het toestaat**. Voor dat laatste stel je één vraag aan de bron zelf: de voorwaardenpagina lezen, zonder
   data. *Pas met de volledige extractie:* **aandeel lege waarden** (80 % leeg bij 10 records heeft een
   95 %-interval van ongeveer 49–94 %), **of de joinsleutel uniek is** (10 unieke id's zeggen niets over
   12.084, want dubbelingen zijn zeldzaam en vallen pas bij grote aantallen op) en **of er genoeg variatie in
   de doelvariabele zit** (opbrengst liep in de proef van $31 k tot $32,7 M, maar 7 waarden zijn geen verdeling).
2. De eerste tien volgen de sortering van de bron, en die is nooit aselect. Drie sorteringen die de
   makkelijkste records naar voren halen: **populariteit** (TMDb `popularity.desc` is de standaard: bekende
   films met volledige data), **relevantie/zoekscore** (Deezer zet het populairste album bovenaan) en
   **id/toevoegdatum** (oudste records zijn het langst bijgewerkt). Het effect is zichtbaar: de populairste film van
   2015 (*Mad Max: Fury Road*, bovenaan bij `popularity.desc`) heeft budget, opbrengst én 13 reviews, terwijl
   onze aselecte steekproef maar 2/10 budgetten had. Wat we deden zonder de limiet los te laten: **tien willekeurige
   pagina's, uit elke pagina één willekeurige film** (seed 42). Die id's hebben we bewaard, zodat de
   steekproef reproduceerbaar is.
3. Bij ons lukte het onderdeel "scraping met login" niet. In één zin: *"Op 23-09-2026 hebben we voor scraping
   met login alleen Letterboxd overwogen. We hebben het niet uitgevoerd, omdat de Terms of Use
   (letterboxd.com/legal/terms-of-use) elke 'robot, spider, scraper' verbieden. Scraping zonder login
   (Wikipedia, 9/10 pagina's, `robots.txt` staat `/wiki/` toe) werkt wel. Het criterium 'scraping' is dus
   gedekt en 'met login' niet. Opnieuw proberen met Letterboxd heeft geen zin; met een andere bron alleen
   na akkoord van de docent."*

**T2. Dekking, joins en vertekening**

1. Inner join: **45 rijen**; één rij = een film uit A die óók in D voorkomt (een deelpopulatie). Left join:
   **380 rijen**; één rij = een film uit A, waarvan 335 (88 %) zonder D-features. Onze eigen proefdata:
   left join 10 rijen, inner join (opbrengst + budget + beeld + audio) **0 rijen**. Bij bekende films
   5/10, dus ≈ 920 van de 1.843.
2. De left join houdt de rijen, maar elke analyse die de D-features gebruikt, laat de lege rijen alsnog
   vallen (listwise deletion) of moet ze invullen. Als de ontbrekende waarden **willekeurig** verdeeld zijn
   (MCAR), heb je alleen minder power. Hangen ze samen met de bron zelf (MNAR), dan analyseer je een scheve
   deelgroep. Bij ons is dat laatste het geval: films **met** bekende opbrengst hebben een mediaan van **290**
   TMDb-stemmen, films **zonder** een mediaan van **100**. Wat overblijft, is de succesvollere helft. Het
   verband met succes wordt dan afgevlakt of zelfs omgekeerd (range restriction, survivorship bias).
3. Vergelijk voor een variabele die bij **alle** films bekend is (vote_count, jaar, genre, populariteit)
   de verdeling tussen films mét en zonder dekking. Dat is de bias-cel in de uitdaging: opbrengst 290 vs 100,
   budget 266 vs 182, review 297 vs 222 stemmen. Een systematisch verschil betekent niet-willekeurige
   ontbrekende waarden. Op de volledige data toetsen we dit met een Mann-Whitney-toets (numeriek) of een
   χ²-toets (genre).
4. Alleen bij de **left join**, en dan alleen voor analyses die de ontbrekende features niet nodig hebben,
   of als je aantoont dat het ontbreken willekeurig is. Dat is bij ons niet zo. Bij de inner join schrijf je
   de populatie expliciet op: *"Deze resultaten gelden voor films uit 2010–2019 met ≥ 1000 TMDb-stemmen
   waarvan budget, opbrengst en een soundtrack op Deezer bekend zijn; ze zijn niet te generaliseren naar
   kleinere films."*

**T3. Granulariteit**

1. TMDb details: 1 rij = 1 film, uniek door `tmdb_id` (`assert df_api["tmdb_id"].is_unique` slaagt).
   TMDb reviews: 1 rij = 1 review, uniek door review-`id`. OMDb: 1 film per `imdb_id`. Wikipedia: 1 pagina
   per film (`tmdb_id`). Wikidata ruw: 1 rij = 1 combinatie film × budgetstelling × box-office-stelling,
   uniek door (film, stelling-id's); na aggregatie 1 rij per `tmdb_id` (assert slaagt). Posters: 1 per
   `tmdb_id`. Deezer: ruw 1 rij per track (album *Emperor*: 17 tracks), na keuze 1 preview per `tmdb_id`.
   In de merge controleert `validate="one_to_one"` dit automatisch voor alle vijf de koppelingen, en die slaagt.
2. De films bepalen de granulariteit van de eindtabel. **Reviews** (1 per review) aggregeren we per film:
   het **gemiddelde** sentiment, plus het **aantal** reviews (een som) als betrouwbaarheidsmaat. Wikidata-
   stellingen per regio aggregeren we naar het **wereldwijde** bedrag of anders het **maximum**, en dus
   géén som, want "VS" zit al in "wereldwijd".
3. Koppel je zonder aggregatie, dan krijg je een **join-explosie** (fan-out, many-to-many): een film met
   8 reviews en 3 box-office-stellingen wordt 24 rijen, en elk gemiddelde telt die film 24× mee.
   Controles die dit meteen laten zien: `assert len(eind) == len(basis)` na elke left join,
   `merge(..., validate="one_to_one")` en `df[sleutel].duplicated().sum()`.
4. Eén poster per film: dat is al één rij. Meerdere fragmenten per film (album van 17 tracks): features
   per track berekenen en daarna per film het **gemiddelde** en de **spreiding** nemen, of bewust één track
   kiezen (de openingstrack of de populairste). Wat je weggooit: de variatie binnen de soundtrack, de
   volgorde en welke track het "thema" is. Bij één preview van 30 s gooi je zelfs de rest van dat nummer weg.

> Eén rij in onze eindtabel is één **film uit 2010–2019 met ≥ 1000 TMDb-stemmen, geïdentificeerd door
> `tmdb_id`**, met daaraan de per film geaggregeerde review-, beeld- en audiofeatures.

**T4. Is de eindtabel groot genoeg?**

1. Met 45 rijen kun je beschrijven (verdelingen, voorbeelden) en alleen **grote** effecten aantonen: een
   verschil tussen twee groepen van ~22 films vanaf d ≈ 0,85, of een correlatie vanaf r ≈ 0,41 (α = 0,05,
   power 0,8). **Niet** mogelijk: een trend over tien jaar (4–5 films per jaar, en dan nog per genre
   uitsplitsen), een voorspellend model met veel features, en kleine effecten zoals r = 0,2 (daarvoor zijn
   194 films nodig).
2. Met 60 features bij 45 rijen zijn er meer parameters dan waarnemingen (p > n). Een lineair model kan de
   data dan **perfect** fitten, met oneindig veel oplossingen, en het leert ruis. Elke extra feature voegt
   een vrijheidsgraad toe waarmee het model toevallige patronen kan volgen. De fit op de trainingsdata wordt
   dus beter en de voorspelling op nieuwe films slechter (overfitting, curse of dimensionality). Vuistregel:
   10–20 waarnemingen per feature, dus 45 rijen dragen hooguit 2–4 features.
3. De drie knoppen, met onze getallen:
   - **Eenheid van analyse** (≥ 50 → ≥ 1000 stemmen): het totaal daalt van 12.084 naar 1.843 films, maar de
     complete rijen stijgen van 0/10 naar 5/10 (≈ 920). De vraag wordt smaller: "bij bekende films".
   - **Bron** (audio laten vallen): in de hoofdsteekproef hebben dan 4/10 films opbrengst + budget + beeld
     (≈ 4.800 films), maar DV5 vervalt.
   - **Type join** (audio als left join): alle 1.843 rijen blijven, maar de audiofeatures ontbreken bij ~50 %.
     De audiovraag wordt een subanalyse op ~920 films.
   - **Onze keuze:** eenheid naar ≥ 1000 stemmen **plus** left join voor audio. Dan hebben we ≈ 920 complete
     films, meer dan de 194 die nodig zijn voor r = 0,2, en alle vijf de deelvragen blijven beantwoordbaar.
4. Nu het goedkoopst: de **eenheid van analyse**. Dat is één filterparameter (`vote_count.gte`) in de
   extractie. In week 14 het duurst: de **bron** wisselen (nieuwe extractiecode, nieuwe matchinglogica, nieuwe
   licentiecheck) en ook de eenheid (alle extracties, posters en previews opnieuw draaien). Het type join is
   technisch altijd goedkoop (één argument), maar verandert wel de claim die je mag doen.

**T5. Een no-go is ook een resultaat**

1. Een **spike** is een kort, tijdgebonden experiment dat alleen onzekerheid wegneemt ("werkt deze bron?"),
   niet een stuk van het eindproduct. De harde tijdsgrens voorkomt dat je blijft doorbouwen aan iets wat
   misschien niet kan; de uitkomst is kennis. "Het is niet gelukt" zegt iets over ons (tijd, kennis). "Het is
   niet haalbaar" zegt iets over de bron, en daar hoort reproduceerbaar bewijs bij: exacte URL, datum,
   statuscode of foutmelding, de tekst van de voorwaarden die het verbieden, of een telvraag die laat zien dat
   er te weinig records zijn (bv. Wikidata: maar 1.134 films met budget én box office).
2. De denkfout is de **sunk-cost fallacy** (verzonken kosten): "we hebben er al twee weken in gestoken". Die
   twee weken zijn weg, wat je ook kiest. De vraag die het besluit weer op de toekomst richt: *"Als we vandaag
   zouden beginnen, met wat we nu weten, zouden we dan deze bron kiezen? Wat kost het vanaf nu om hem werkend
   te krijgen, vergeleken met het alternatief?"*
3. Per bron (alles gecontroleerd op 23-09-2026):

   | Bron | Gebruiksvoorwaarden | Licentie | AVG |
   |---|---|---|---|
   | TMDb (+ posters) | [API Terms of Use](https://www.themoviedb.org/api-terms-of-use): niet-commercieel, attributie verplicht, ML-training = commercieel | geen open licentie; posters © rechthebbenden | reviews bevatten gebruikersnamen → niet opgeslagen |
   | OMDb | — | **CC BY-NC 4.0** ([omdbapi.com](https://www.omdbapi.com/)) | n.v.t. |
   | Wikipedia | [robots.txt](https://en.wikipedia.org/robots.txt): `/wiki/` toegestaan; [User-Agent-beleid](https://meta.wikimedia.org/wiki/User-Agent_policy) | tekst **CC BY-SA 4.0**; wij nemen alleen feiten over | n.v.t. |
   | Wikidata | query-limiet 60 s; User-Agent-beleid | **CC0 1.0** ([Wikidata:Licensing](https://www.wikidata.org/wiki/Wikidata:Licensing)) | n.v.t. |
   | Deezer | [Terms of Use](https://developers.deezer.com/termsofuse): art. IV strikt niet-commercieel; art. V geen maatregelen omzeilen om Content te downloaden | muziek © labels/componisten | n.v.t. |
   | Letterboxd | [Terms of Use](https://letterboxd.com/legal/terms-of-use/): geen robot/spider/scraper → **no-go** | — | reviews = persoonsgegevens |

   Let op: onze opdrachtgever is een (fictieve) filmproducent, dus een commerciële partij. TMDb, OMDb
   (NC) en Deezer staan alleen niet-commercieel gebruik toe. Voor dit schoolproject is dat in orde, maar
   de producent zou de pipeline niet zonder commerciële licentie kunnen gebruiken. Dat hoort in het PvA.
4. Een bron die niet mag, kun je net zo min in je eindproduct gebruiken als een bron die niet werkt. Het
   resultaat mag niet gepubliceerd of ingeleverd worden, en je loopt juridisch risico. Het is een harde
   randvoorwaarde, geen technisch probleem dat je met meer uren oplost. In de kolom *blokkade* schrijf je de
   juridische reden met een verwijzing: *"Juridisch: Letterboxd ToS verbiedt scrapers
   (letterboxd.com/legal/terms-of-use, gelezen 23-09-2026) → no-go, alternatief: …"*.

**T6. Power en causaliteit**

1. Als je de kleinste relevante effectgrootte achteraf uit je eigen data afleest, kies je hem (bewust of niet)
   zo dat de uitkomst "significant" of "voldoende power" wordt. Dan is de redenering rond (HARKing,
   p-hacking; post-hoc power volgt rechtstreeks uit de p-waarde). Vooraf vastleggen maakt de toets eerlijk
   en bepaalt hoeveel data je moet verzamelen. Halveren van de effectgrootte geeft **4× zoveel n**:
   d = 0,5 → 63 per groep, d = 0,25 → 252 per groep; r = 0,2 → 194, r = 0,1 → 783 (V1).
2. Corrigeren voor (of selecteren op) een **collider** opent een pad dat eerst dicht was. In onze DAG:
   budget → *in_bron* ← opbrengst. Onder de films die wél in de bron staan, zijn goedkope films vooral
   aanwezig als ze een hit waren. Daardoor ontstaat een kunstmatig verband tussen budget en opbrengst. Een
   inner join op "heeft bedragen/soundtrack" doet precies dit, en de V2-code toont dat dit pad dan opengaat.
   Corrigeren voor een **mediator** blokkeert het deel van het effect dat via die mediator loopt: met
   *release_breedte* in het model meet je alleen het directe effect van budget, niet het totale.
   Per variabele uit onze DAG:

   | Variabele | Corrigeren? | Reden |
   |---|---|---|
   | genre | **ja** | confounder: backdoorpad budget ← genre → opbrengst |
   | jaar | **ja** | confounder (inflatie, streamingtijdperk) |
   | franchise | **ja** | confounder: vervolgfilms krijgen meer budget én hebben een vast publiek |
   | release_breedte | **nee** | mediator: we willen het totale effect van budget |
   | in_bron | **nee** | collider: niet op selecteren. Daarom audio als left join en de populatie expliciet benoemen |
   | verwachte kwaliteit | kan niet | ongemeten confounder → alleen uitspraken over samenhang |

---

## Controle van je uitwerking

1. Vul de haalbaarheidstabel in deel 6 volledig in.
2. Zorg dat de proefbestanden in `data/proef/` staan.
3. Verwerk de uitkomsten van de proef in de haalbaarheidstabel bij het PvA.
4. Zorg dat je de proef tijdens het go/no-go-gesprek kunt demonstreren; het notebook
   zelf hoef je niet in te leveren.

## Wat neem je hiervan mee naar je portfolio?

* De haalbaarheidstabel gaat als bijlage bij je plan van aanpak.
* De code per bron is de **eerste versie** van je extract-laag. In week 15 maak je er
  nette, herbruikbare functies van — maar de moeilijkste vraag ("werkt deze bron
  überhaupt?") heb je dan al beantwoord.
* De rekensom van je eindtabel bepaalt of je vraag beantwoordbaar is. Zet hem in je PvA.